# 14 · Efficient Training and Scaling

**Part VII — Engineering, trust and practice** · Estimated time: ~120 min

Up to now we have treated training time and memory as facts of life: a run took as long as it took,
and if the batch did not fit on the GPU we made it smaller. In this notebook we turn those facts into
quantities we can *measure*, *predict* and *engineer*. We start by finding out where the time in a
training step actually goes, then do the memory arithmetic that tells us in advance whether a model
will fit, and then work through the toolbox that practitioners use to train faster and larger:
mixed precision, gradient accumulation, activation checkpointing, `torch.compile`, automated
hyperparameter search with Optuna, and distributed data parallelism. We close with the scaling laws
that guide how compute should be split between model size and data, and with the unglamorous but
decisive discipline of configuration-driven, logged experiments.

## Learning objectives

- Time GPU and CPU code correctly (warm-up, synchronization, medians) and read a `torch.profiler` table.
- Estimate the memory of a training run from parameter counts, optimizer choice and activation sizes, and verify the estimate empirically.
- Explain float16 versus bfloat16, derive why fp16 needs loss scaling, and train with `torch.amp.autocast` and `torch.amp.GradScaler`.
- Implement gradient accumulation and activation checkpointing correctly and quantify their trade-offs.
- Use `torch.compile` and TF32 safely, and know their failure modes.
- Run a pruned Optuna study and interpret optimization history, parameter importance and parallel-coordinate plots.
- Describe data, tensor and pipeline parallelism, DDP mechanics and FSDP/ZeRO sharding, and write a complete `torchrun`-launchable DDP script.
- Apply the $6ND$ compute rule and the Chinchilla heuristic, and run configuration-driven experiments with JSON logs.

## Prerequisites

- Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization (optimizers, schedules, BatchNorm).
- Notebook 05 · The PyTorch Workflow and Best Practices (training loops, DataLoaders, checkpoints, configs).
- Notebook 06 · Convolutional Neural Networks (the small CNN we profile here).
- Notebook 09 · Attention and Transformers (parameter and FLOP counting of a Transformer block; scaling in prose).

## Contents

1. Where the time goes: timing, profiling and data loading
2. Memory arithmetic
3. Mixed precision
4. Fitting bigger models: accumulation, checkpointing, 8-bit optimizers, `channels_last`
5. `torch.compile`, TF32 and matmul precision
6. Hyperparameter optimization with Optuna
7. Distributed training concepts and a DDP script
8. Scaling laws, compute budgets and experiment hygiene
- Exercises · Summary · Further reading

In [ ]:
import contextlib, copy, io, json, math, os, random, time, textwrap
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from torchvision.transforms import v2

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
print(f"CPU threads used by PyTorch: {torch.get_num_threads()}")

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

# Optional libraries used later in the notebook. Each is guarded so the notebook still runs without it.
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAVE_OPTUNA = True
    print(f"optuna {optuna.__version__} available")
except ImportError:
    HAVE_OPTUNA = False
    print("optuna is not installed: Section 6 will print a message instead of running the study. "
          "Create the conda environment from environment.yml to get it.")

try:
    from torch.profiler import profile, ProfilerActivity, record_function
    HAVE_PROFILER = True
except Exception as e:  # extremely unusual, but keep the notebook runnable
    HAVE_PROFILER = False
    record_function = lambda name: contextlib.nullcontext()
    print("torch.profiler is unavailable:", e)

## 1. Where the time goes: timing, profiling and data loading

### 1.1 Measuring time correctly

A training step is a pipeline with several stages: the data loader produces a batch (CPU work), the
batch is copied to the accelerator, the forward pass runs, the backward pass runs, and the optimizer
updates the parameters. Speeding up a stage that takes 3 % of the time cannot give more than a 3 %
speed-up (this is **Amdahl's law**), so the first job is always to measure which stage dominates. That
sounds trivial, but on a GPU it is surprisingly easy to measure the wrong thing.

The reason is that CUDA execution is **asynchronous**. When Python executes `c = a @ b` on CUDA tensors,
PyTorch only *enqueues* a kernel on a stream and returns immediately; the multiplication happens later,
whenever the GPU gets to it. A naive `time.perf_counter()` before and after the line therefore measures
the cost of launching the kernel (a few microseconds), not the cost of running it. To measure the real
duration we must call `torch.cuda.synchronize()`, which blocks the CPU until every queued kernel has
finished. Three further rules make timings trustworthy:

1. **Warm up** first. The first call of an operation pays one-off costs: CUDA context creation, cuDNN
   autotuning of convolution algorithms, memory-allocator growth, and (for compiled code) compilation.
2. **Repeat and take the median**, not the mean: a single background process can add a millisecond
   outlier that would dominate a mean of a few runs.
3. **Avoid hidden synchronizations** inside the timed region unless they are part of the workload.
   `loss.item()`, `tensor.cpu()`, `print(tensor)` and `if tensor > 0:` all force a synchronization,
   which stalls the GPU pipeline. In training loops we should accumulate metrics on the device and call
   `.item()` once per epoch.

The helper below encodes these rules. Its `sync()` is a no-op on the CPU, so the same function gives
correct results on both devices.

In [ ]:
def sync():
    """Block until all queued device work has finished (a no-op on the CPU)."""
    if device.type == "cuda":
        torch.cuda.synchronize()


def time_fn(fn, n_warmup=2, n_repeat=5):
    """Median wall-clock seconds of one call of fn(), after warm-up and with proper synchronization."""
    for _ in range(n_warmup):
        fn()
    sync()
    times = []
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        fn()
        sync()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


n = 2048 if device.type == "cuda" else 512
a = torch.randn(n, n, device=device)
b = torch.randn(n, n, device=device)

# The naive measurement: no synchronization after the operation.
for _ in range(2):
    _ = a @ b
sync()
t0 = time.perf_counter()
c = a @ b
t_naive = time.perf_counter() - t0
sync()

t_correct = time_fn(lambda: a @ b)
gflop = 2 * n ** 3 / 1e9
print(f"{n}x{n} matmul on {device.type}: naive timing {t_naive * 1e3:8.3f} ms | "
      f"synchronized median {t_correct * 1e3:8.3f} ms | {gflop / t_correct:8.1f} GFLOP/s")

On a GPU the naive number is typically a few tens of microseconds regardless of the matrix size, while
the synchronized one grows with $n^3$; the gap is the whole point of the demonstration. On a CPU both
numbers agree because CPU kernels run synchronously. The last column, achieved GFLOP/s, is worth
remembering as a yardstick: a laptop CPU reaches tens to a few hundred GFLOP/s in float32, a modern
data-centre GPU reaches tens of TFLOP/s in float32 and hundreds in float16/bfloat16 with tensor cores.
Whenever our code achieves a small fraction of that peak, something other than arithmetic is the
bottleneck, and the profiler in Section 1.3 will tell us what.

> **Common pitfall.** `time.time()` around a CUDA operation without `torch.cuda.synchronize()` measures
> kernel *launch* time. Any benchmark that reports a GPU running a large matmul in 20 µs is wrong.

### 1.2 The running example: FashionMNIST in memory

Throughout this notebook we use FashionMNIST (70 000 grey-scale 28×28 images in ten clothing classes,
introduced in Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization).
To keep the focus on the training mechanics rather than on I/O, we load the
whole dataset into memory tensors *once* and move them to the device, so that a mini-batch is nothing
more than a tensor slice. This is a legitimate and common technique for small datasets: the full
training set is 60 000 × 784 float32 values, about 180 MB, which any GPU can hold. We will measure in
Section 1.4 exactly how much time this saves compared with a per-sample `DataLoader`.

We also define the two model families that reappear in later sections: a configurable MLP and a
small CNN with BatchNorm. Under `FAST_RUN` we use a 6 000-image training subset.

In [ ]:
def load_fashion_mnist(train, n=None):
    """Return (X, y) as normalized float tensors of shape (N, 1, 28, 28) and (N,)."""
    ds = datasets.FashionMNIST(DATA_DIR, train=train, download=True)
    X = ds.data.float().div(255.0).unsqueeze(1)        # uint8 (N, 28, 28) -> float (N, 1, 28, 28) in [0, 1]
    y = ds.targets.clone()
    if n is not None:
        X, y = X[:n], y[:n]
    X = (X - 0.2860) / 0.3530                          # dataset mean and std
    return X, y


N_TRAIN = 6_000 if FAST_RUN else 60_000
N_VAL = 2_000 if FAST_RUN else 10_000
X_train, y_train = load_fashion_mnist(train=True, n=N_TRAIN)
X_val, y_val = load_fashion_mnist(train=False, n=N_VAL)
X_train, y_train, X_val, y_val = [t.to(device) for t in (X_train, y_train, X_val, y_val)]
print("train:", tuple(X_train.shape), "val:", tuple(X_val.shape),
      f"| {X_train.numel() * 4 / 2**20:.0f} MB of training pixels on {device}")


class MLP(nn.Module):
    """Fully connected classifier for 28x28 images: depth hidden layers of the same width."""

    def __init__(self, hidden=256, depth=2, dropout=0.0, n_classes=10):
        super().__init__()
        layers, d_in = [], 28 * 28
        for _ in range(depth):
            layers += [nn.Linear(d_in, hidden), nn.ReLU(), nn.Dropout(dropout)]
            d_in = hidden
        layers.append(nn.Linear(d_in, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.flatten(1))


class SmallCNN(nn.Module):
    """Two conv-BN-ReLU-pool blocks followed by a two-layer head."""

    def __init__(self, width=32, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, width, 3, padding=1), nn.BatchNorm2d(width), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(width, 2 * width, 3, padding=1), nn.BatchNorm2d(2 * width), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(2 * width * 7 * 7, 128), nn.ReLU(),
                                  nn.Linear(128, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters())


print(f"MLP params: {count_params(MLP()):,} | SmallCNN params: {count_params(SmallCNN()):,}")

The training and evaluation helpers follow the pattern of Notebook 05 · The PyTorch Workflow and Best
Practices, with one change that matters for
this notebook: the running loss is accumulated *on the device* as a tensor and converted with `.item()`
only once per epoch. On a GPU, calling `.item()` after every step would force a synchronization per
step and stall the pipeline; on the CPU it makes no difference, but the habit is worth acquiring.
`iterate_minibatches` slices the in-memory tensors directly instead of going through a `DataLoader`.

In [ ]:
def iterate_minibatches(X, y, batch_size, shuffle=True):
    """Yield (xb, yb) slices of in-memory tensors; shuffling uses torch's global RNG."""
    n = X.shape[0]
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    for start in range(0, n, batch_size):
        j = idx[start:start + batch_size]
        yield X[j], y[j]


def train_one_epoch(model, opt, X, y, batch_size=128, scheduler=None, max_steps=None):
    """One pass over (X, y); returns the mean training loss. Metrics stay on the device until the end."""
    model.train()
    loss_sum = torch.zeros((), device=X.device)
    n_seen, steps = 0, 0
    for xb, yb in iterate_minibatches(X, y, batch_size):
        loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if scheduler is not None:
            scheduler.step()
        loss_sum += loss.detach() * xb.shape[0]
        n_seen += xb.shape[0]
        steps += 1
        if max_steps is not None and steps >= max_steps:
            break
    return loss_sum.item() / n_seen


@torch.no_grad()
def evaluate(model, X, y, batch_size=1000):
    """Mean cross-entropy and accuracy on (X, y)."""
    model.eval()
    loss_sum = torch.zeros((), device=X.device)
    correct = torch.zeros((), device=X.device)
    for xb, yb in iterate_minibatches(X, y, batch_size, shuffle=False):
        logits = model(xb)
        loss_sum += F.cross_entropy(logits, yb, reduction="sum")
        correct += (logits.argmax(1) == yb).sum()
    return loss_sum.item() / X.shape[0], correct.item() / X.shape[0]


EPOCHS_BASE = 1 if FAST_RUN else 3
torch.manual_seed(SEED)
cnn = SmallCNN().to(device)
opt = torch.optim.SGD(cnn.parameters(), lr=0.05, momentum=0.9)
for epoch in range(EPOCHS_BASE):
    t0 = time.perf_counter()
    tr = train_one_epoch(cnn, opt, X_train, y_train)
    vl, va = evaluate(cnn, X_val, y_val)
    print(f"epoch {epoch + 1} | train loss {tr:.3f} | val loss {vl:.3f} | val acc {va:.3f} | {time.perf_counter() - t0:.1f}s")

The baseline confirms that the harness works (roughly 80–85 % validation accuracy after one epoch on
the subset, around 90 % after three full epochs) and gives us a first number: the seconds per epoch
printed above. Everything in the rest of the notebook is about understanding and reducing that number,
or about training something bigger in the same time.

### 1.3 Reading a `torch.profiler` trace

A timer tells us *how long*; a profiler tells us *what*. `torch.profiler` records every operator call
(`aten::conv2d`, `aten::addmm`, ...) with its duration on the CPU and, if present, on the GPU, and can
also track memory and input shapes. The typical workflow is:

1. warm up the step a few times outside the profiler,
2. profile a handful of steps inside `with profile(...) as prof:`,
3. print `prof.key_averages().table(...)` sorted by the time you care about,
4. optionally export a Chrome trace (`prof.export_chrome_trace("trace.json")`) and open it in
   `chrome://tracing` or Perfetto to see the timeline of kernels, gaps and synchronizations.

`record_function("name")` lets us label regions of our own code so that they appear as rows in the
table. We profile three full training steps of the small CNN on a batch of 128 images.

In [ ]:
def train_step(model, opt, xb, yb):
    """A labelled training step for profiling."""
    model.train()
    with record_function("forward"):
        loss = F.cross_entropy(model(xb), yb)
    with record_function("backward"):
        opt.zero_grad(set_to_none=True)
        loss.backward()
    with record_function("optimizer_step"):
        opt.step()
    return loss


def profile_table(prof, row_limit=15):
    """key_averages() table sorted by device time when a GPU is present, else by self CPU time."""
    keys = (["self_device_time_total", "self_cuda_time_total"] if device.type == "cuda" else []) + ["self_cpu_time_total"]
    for key in keys:
        try:
            return prof.key_averages().table(sort_by=key, row_limit=row_limit)
        except Exception:
            continue
    return prof.key_averages().table(row_limit=row_limit)


torch.manual_seed(SEED)
prof_model = SmallCNN().to(device)
prof_opt = torch.optim.SGD(prof_model.parameters(), lr=0.05, momentum=0.9)
xb, yb = X_train[:128], y_train[:128]
for _ in range(3):                       # warm-up outside the profiler
    train_step(prof_model, prof_opt, xb, yb)
sync()

if HAVE_PROFILER:
    activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device.type == "cuda" else [])
    try:
        with profile(activities=activities, record_shapes=True, profile_memory=True) as prof:
            for _ in range(3):
                train_step(prof_model, prof_opt, xb, yb)
            sync()
        print(profile_table(prof))
        trace_path = DATA_DIR / "profiler_trace_cnn_step.json"
        prof.export_chrome_trace(str(trace_path))
        print(f"Chrome trace written to {trace_path} (open it in chrome://tracing or https://ui.perfetto.dev)")
    except Exception as e:
        print("torch.profiler could not run in this environment:", type(e).__name__, str(e)[:200])
else:
    print("torch.profiler unavailable; skipping the profile.")

How to read the table. Each row is one operator aggregated over the three steps. **Self CPU** is the
time spent inside that operator excluding its children; **CPU total** includes children (the row
`backward` is a parent of everything the autograd engine ran). **# of Calls** shows how many times the
operator ran; a row with a small self time but thousands of calls is a per-call overhead problem, which
is exactly what the optimizer step of a model with many small parameter tensors looks like (dozens of
tiny `aten::add_`/`aten::mul_` calls per step; `torch.optim` offers `foreach=True` and `fused=True`
implementations to batch them). On the GPU the important columns are the device-time ones, and it is
common to see the CPU busy launching kernels while the GPU is mostly idle: that is a launch-bound
workload, and the fix is larger batches, fewer smaller ops, CUDA graphs or `torch.compile`
(Section 5).

For this CNN you should see the convolution forward and backward kernels (`aten::convolution`,
`aten::convolution_backward`, or the mkldnn/cuDNN kernels beneath them) at the top, followed by the
linear layers (`aten::addmm`) and the BatchNorm kernels. The memory columns (`CPU Mem`, `Self CPU Mem`
or the CUDA equivalents) show which operators allocate the activation tensors that Section 2 accounts
for analytically.

> **Try it.** Change the batch to 8 images and re-run. The convolution rows shrink far more than the
> optimizer rows, which do not depend on the batch size at all: at small batches the update becomes a
> fixed cost that dominates.

### 1.4 Data loading: where a surprising amount of time hides

Data loading is single-threaded Python unless we ask otherwise, and Python is slow at the granularity
of one image. A standard `DataLoader` calls `dataset[i]` once *per sample*, applies the transform
pipeline to each sample, and then stacks the results with `collate_fn`. For a 28×28 image decoded from
PIL and normalized, that is tens of microseconds per image, which for a batch of 128 adds up to
several milliseconds, comparable to the forward pass of a small model. Three progressively faster
alternatives are:

- **`TensorDataset` + `DataLoader`**: no decoding, but still one Python `__getitem__` call and one
  `torch.stack` per batch.
- **Tensor slicing** (our `iterate_minibatches`): one fancy-indexing kernel per batch, no per-sample
  Python at all.
- **Preprocessing on the GPU**: if the data lives on the device, normalization and even augmentations
  (random crops via `torch.roll`/slicing, flips via `torch.flip`, noise) can be applied to the whole
  batch with a handful of kernels.

Let us measure the first two against tensor slicing. We time eight batches of 128 with each method.

In [ ]:
raw_train = datasets.FashionMNIST(
    DATA_DIR, train=True, download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize([0.2860], [0.3530])]),
)
loader_pil = DataLoader(raw_train, batch_size=128, shuffle=True, num_workers=0, pin_memory=(device.type == "cuda"))
loader_tensor = DataLoader(TensorDataset(X_train.cpu(), y_train.cpu()), batch_size=128, shuffle=True,
                           num_workers=0, pin_memory=(device.type == "cuda"))
N_BATCHES = 8


def take_batches(loader, n):
    it = iter(loader)
    for _ in range(n):
        xb, yb = next(it)
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
    sync()


def slice_batches(n):
    for k, (xb, yb) in enumerate(iterate_minibatches(X_train, y_train, 128)):
        if k + 1 >= n:
            break
    sync()


t_pil = time_fn(lambda: take_batches(loader_pil, N_BATCHES), n_warmup=1, n_repeat=3) / N_BATCHES
t_tensor = time_fn(lambda: take_batches(loader_tensor, N_BATCHES), n_warmup=1, n_repeat=3) / N_BATCHES
t_slice = time_fn(lambda: slice_batches(N_BATCHES), n_warmup=1, n_repeat=3) / N_BATCHES
for name, t in [("DataLoader over PIL images + transforms", t_pil),
                ("DataLoader over TensorDataset", t_tensor),
                ("direct tensor slicing (on device)", t_slice)]:
    print(f"{name:42s} {t * 1e3:8.2f} ms / batch of 128  ({t / 128 * 1e6:6.1f} µs per sample)")

The per-sample pipeline is typically one to two orders of magnitude slower than slicing a device
tensor, and for a small model this data time can exceed the compute time of the step. For real image
datasets that do not fit in memory, the knobs are the following (explained rather than measured, because
worker processes are unreliable inside notebooks on Windows and we keep `num_workers=0` throughout the
course):

- **`num_workers=k`** runs `k` copies of the dataset in separate processes, each producing whole
  batches; the main process only receives finished tensors. Start with the number of physical cores and
  measure; more workers than cores hurts.
- **`pin_memory=True`** allocates the batch in page-locked host memory, which the GPU can read by DMA;
  together with `x.to(device, non_blocking=True)` the copy overlaps with computation.
- **`persistent_workers=True`** keeps the workers alive between epochs instead of re-spawning them
  (spawning is expensive on Windows and macOS). **`prefetch_factor`** (default 2) is how many batches
  each worker prepares ahead.
- **Decode once, store tensors.** If the dataset fits in RAM as uint8 (a 60 000 × 28 × 28 array is only
  47 MB; even CIFAR-10 is 150 MB), decode it once and keep the transform pipeline to cheap tensor ops.
- **GPU-side augmentation.** Random crops, flips and normalization on the whole batch cost a few
  kernels; libraries such as NVIDIA DALI take this idea to JPEG decoding on the GPU.

The last stage of the input pipeline is the host-to-device copy. Its bandwidth depends on whether the
source memory is pinned; the next cell measures it when a GPU is present and otherwise explains what
would be measured.

In [ ]:
if device.type == "cuda":
    xb_pageable = X_train[:1024].cpu().contiguous()
    xb_pinned = xb_pageable.pin_memory()
    t_pageable = time_fn(lambda: xb_pageable.to(device))
    t_pinned = time_fn(lambda: xb_pinned.to(device, non_blocking=True))
    mb = xb_pageable.numel() * 4 / 2**20
    print(f"copy of {mb:.1f} MB: pageable {t_pageable * 1e3:.3f} ms ({mb / t_pageable / 1024:.1f} GB/s) | "
          f"pinned + non_blocking {t_pinned * 1e3:.3f} ms ({mb / t_pinned / 1024:.1f} GB/s)")
else:
    print("CPU-only session: there is no host-to-device copy to measure.")
    print("On a GPU, a pageable copy first moves the data into a pinned staging buffer and then across PCIe "
          "(roughly 3-12 GB/s), while a pinned copy goes straight to the GPU and can overlap with compute "
          "when non_blocking=True is used.")

On a GPU, pinned copies are usually 1.5–3× faster and, more importantly, asynchronous: the next batch
can stream in while the current one is being processed. For our in-memory approach the copy happens
once for the whole dataset, so it disappears from the per-step cost entirely.

> **Key idea.** Measure before optimizing. A synchronized timer and one profiler table tell you whether
> a step is data-bound, launch-bound, memory-bound or compute-bound, and each of these has a different
> remedy.

## 2. Memory arithmetic

### 2.1 What occupies device memory during training

Out-of-memory errors are the most common way a training run dies, and they are also the most
predictable. The memory used by a training step decomposes into a part that depends only on the model
and optimizer (the **training state**) and a part that scales with the batch size (the **activations**).

Let $P$ be the number of parameters and let each parameter occupy $b_p$ bytes (4 for float32). The
training state consists of

- the parameters themselves: $P b_p$ bytes;
- their gradients, same shape and dtype: $P b_p$ bytes;
- the optimizer state: SGD keeps nothing, SGD with momentum keeps one buffer ($4P$ bytes with float32
  parameters), Adam and AdamW keep two (first and second moment, $8P$ bytes).

With float32 parameters and Adam this gives the well-known figure of **16 bytes per parameter**: a
7-billion-parameter model needs 112 GB before a single activation is stored, which is why such models
cannot be trained on one 80 GB GPU without sharding (Section 7). Mixed precision (Section 3) does *not*
reduce this figure much: parameters and gradients shrink to 2 bytes each, but a float32 master copy
(4 bytes) is added and the Adam moments stay in float32, giving $2 + 2 + 4 + 8 = 16$ bytes again. What
mixed precision halves is the activation memory, which is where the batch size enters.

Activations are every intermediate tensor that autograd must keep for the backward pass. For a linear
layer $\mathbf{h} = \mathbf{W}\mathbf{x} + \mathbf{b}$ the backward pass needs $\mathbf{x}$ (to compute
$\partial L / \partial \mathbf{W} = \boldsymbol{\delta}\,\mathbf{x}^\top$); for a ReLU it needs the sign
of its input; for a convolution it needs its input feature map; for BatchNorm it needs the normalized
input and the batch statistics. A useful first-order rule is therefore

$$
M_{\text{act}} \;\approx\; B \cdot b_a \cdot \sum_{\ell} n_\ell ,
$$

where $B$ is the batch size, $b_a$ the bytes per activation element (4 in float32, 2 under autocast)
and $n_\ell$ the number of output elements of layer $\ell$ for one sample. For an MLP the sum is just the
sum of layer widths; for a CNN it is dominated by the early, high-resolution feature maps
($C \times H \times W$ per sample); for a Transformer it is dominated by the attention scores,
$a \cdot s^2$ per layer and sample for $a$ heads and sequence length $s$.

We now write an estimator based on this rule, then measure the truth.

In [ ]:
OPT_STATE_FLOATS = {"sgd": 0, "sgd_momentum": 1, "adam": 2, "adamw": 2}


def training_state_bytes(n_params, optimizer="adam", param_bytes=4, master_fp32=False):
    """Bytes for parameters + gradients + optimizer state (optimizer state is kept in float32)."""
    per_param = 2 * param_bytes + 4 * OPT_STATE_FLOATS[optimizer]
    if master_fp32:
        per_param += 4
    return n_params * per_param


def activation_elements_per_sample(model, x1):
    """Sum of the output elements of every leaf module for a single input sample (rule-of-thumb proxy)."""
    counts, hooks = [], []
    for m in model.modules():
        if len(list(m.children())) == 0:
            hooks.append(m.register_forward_hook(lambda mod, inp, out: counts.append(out.numel())))
    model.eval()
    with torch.no_grad():
        model(x1)
    for h in hooks:
        h.remove()
    return sum(counts)


def estimate_train_memory(model, x1, batch_size, optimizer="adam", dtype_bytes=4):
    P = count_params(model)
    state = training_state_bytes(P, optimizer, dtype_bytes)
    acts = activation_elements_per_sample(model, x1) * batch_size * dtype_bytes
    return {"params": P, "state_bytes": state, "activation_bytes": acts, "total_bytes": state + acts}


MB = 2**20
x1 = X_train[:1]
for name, m in [("MLP(256, depth 2)", MLP().to(device)), ("SmallCNN(width 32)", SmallCNN().to(device))]:
    for B in (32, 128, 512):
        est = estimate_train_memory(m, x1, B)
        print(f"{name:20s} B={B:4d} | params {est['params']:9,d} | state {est['state_bytes'] / MB:7.1f} MB | "
              f"activations {est['activation_bytes'] / MB:7.1f} MB | total {est['total_bytes'] / MB:7.1f} MB")

Two things stand out. For the MLP the state dominates at every reasonable batch size, because a
784×256 weight matrix is large while the activations are only about 1 500 floats per sample. For the
CNN the picture is reversed: the first block produces a 32×28×28 = 25 088-element feature map per
sample, 32 times the number of elements in the input (and the block keeps three such maps: the outputs
of the convolution, of BatchNorm and of ReLU), so at batch 512 the activations dwarf the state. This is the general pattern: **convolutional and Transformer models are activation-bound,
embedding-heavy and MLP models are state-bound**, and the two regimes call for different remedies
(smaller batches, checkpointing and mixed precision for the first; sharding and 8-bit optimizers for
the second).

### 2.2 Measuring the truth

The rule of thumb over-counts some tensors (a `Dropout(p=0)` or a `Flatten` output is not a new
tensor) and misses others (BatchNorm saves statistics, the loss saves its softmax), so we want a
measurement. Two are available:

- On a GPU, `torch.cuda.max_memory_allocated()` after `torch.cuda.reset_peak_memory_stats()` gives
  the peak of the caching allocator, which includes activations, gradients and temporaries.
- On any device, `torch.autograd.graph.saved_tensors_hooks` lets us register a *pack hook* that autograd
  calls for every tensor it saves for backward. Summing the storage sizes of those tensors, excluding
  parameters and counting each storage once, gives exactly the activation memory autograd holds.

The second tool is the more instructive one, because it counts precisely the quantity our formula tries
to predict. We wrap it in a small context manager.

In [ ]:
class SavedTensorCounter:
    """Context manager that sums the bytes of every distinct tensor storage autograd saves for backward,
    excluding the model parameters. Works on CPU and GPU."""

    def __init__(self, model):
        self.skip = {p.untyped_storage().data_ptr() for p in model.parameters()}
        self.seen, self.bytes, self.n_saved = set(), 0, 0

    def _pack(self, t):
        if isinstance(t, torch.Tensor):
            self.n_saved += 1
            st = t.untyped_storage()
            ptr = st.data_ptr()
            if ptr not in self.skip and ptr not in self.seen:
                self.seen.add(ptr)
                self.bytes += st.nbytes()
        return t

    @staticmethod
    def _unpack(t):
        return t

    def __enter__(self):
        self.seen, self.bytes, self.n_saved = set(), 0, 0
        self._ctx = torch.autograd.graph.saved_tensors_hooks(self._pack, self._unpack)
        self._ctx.__enter__()
        return self

    def __exit__(self, *exc):
        self._ctx.__exit__(*exc)
        return False


def measure_step_memory(model_fn, batch_size, optimizer="adam"):
    """Run one Adam/SGD step at the given batch size; return measured activation bytes (and GPU peak)."""
    torch.manual_seed(SEED)
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        base = torch.cuda.memory_allocated()
    model = model_fn().to(device)
    opt = torch.optim.Adam(model.parameters()) if optimizer == "adam" else torch.optim.SGD(model.parameters(), lr=0.1)
    xb, yb = X_train[:batch_size].clone(), y_train[:batch_size].clone()   # fresh storage: a slice is a view of the whole dataset tensor
    model.train()
    for step in range(2):                    # step 0 creates the optimizer state, step 1 is measured
        with SavedTensorCounter(model) as counter:
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
        opt.step()
    result = {"saved_activation_bytes": counter.bytes, "n_saved_tensors": counter.n_saved}
    if device.type == "cuda":
        torch.cuda.synchronize()
        result["gpu_peak_bytes"] = torch.cuda.max_memory_allocated() - base
    return result


print(f"{'model':20s} {'B':>4s} {'est. activations':>17s} {'measured (autograd)':>20s} {'est. total':>11s} {'GPU peak':>10s}")
for name, fn in [("MLP(256, depth 2)", MLP), ("SmallCNN(width 32)", SmallCNN)]:
    for B in (32, 128, 512):
        est = estimate_train_memory(fn().to(device), x1, B)
        meas = measure_step_memory(fn, B)
        peak = f"{meas['gpu_peak_bytes'] / MB:8.1f} MB" if "gpu_peak_bytes" in meas else "   (CPU)"
        print(f"{name:20s} {B:4d} {est['activation_bytes'] / MB:14.1f} MB {meas['saved_activation_bytes'] / MB:17.1f} MB "
              f"{est['total_bytes'] / MB:8.1f} MB {peak}")

The measured activation memory grows linearly in the batch size, as the formula predicts, and the
constant is of the same order as the estimate; the estimate is a little high for the MLP (it counts the
`Dropout` and the second `ReLU` outputs which share storage with tensors already counted) and closer
for the CNN, where the big feature maps dominate everything. On a GPU the peak column exceeds the sum of
state and saved activations because the backward pass allocates gradient buffers and temporaries while
the activations are still alive, and because the caching allocator rounds allocations up; a safety
factor of 1.2–1.5 on top of the estimate is a good planning rule.

> **Try it.** Replace `torch.optim.Adam` with SGD in `measure_step_memory` (pass `optimizer="sgd"`) and
> compare the GPU peak: the difference is exactly $8P$ bytes, the two Adam moment buffers.

### 2.3 The same arithmetic for Transformers

For the models that actually run out of memory in practice, the same formulas apply, with a standard
parameter count per Transformer block of $12 d^2$ (four $d \times d$ attention projections and an MLP
with $8 d^2$ weights, cf. Notebook 09 · Attention and Transformers) plus embeddings. Activation memory
per block for a batch of $b$ sequences of length $s$ with hidden size $h$ and $a$ heads is approximately
$s\,b\,h\,(34 + 5\,a\,s / h)$ bytes in half precision without any recomputation (Korthikanti et al.,
2022). The second term is the attention scores and their softmax, quadratic in the sequence length,
which is the reason long-context training relies on FlashAttention-style kernels that never
materialize them. The table below applies both formulas to a few well-known model sizes.

In [ ]:
def transformer_params(n_layers, d_model, vocab, seq_len):
    per_layer = 12 * d_model ** 2 + 13 * d_model              # weights + biases + LayerNorm gains/biases
    return n_layers * per_layer + vocab * d_model + seq_len * d_model


def transformer_activation_bytes(n_layers, d_model, n_heads, seq_len, batch, bytes_per_elem=2):
    per_layer = seq_len * batch * d_model * (34 + 5 * n_heads * seq_len / d_model) * bytes_per_elem / 2
    return n_layers * per_layer                               # the formula is stated for 2-byte elements


GB = 2**30
configs = [("GPT-2 small", 12, 768, 12, 50257, 1024, 8),
           ("GPT-2 XL", 48, 1600, 25, 50257, 1024, 8),
           ("7B-class LLM", 32, 4096, 32, 32000, 4096, 1)]
print(f"{'model':14s} {'params':>8s} {'Adam state (fp32)':>18s} {'activations/step':>17s}  (batch x seq)")
for name, L, d, a, V, s, b in configs:
    P = transformer_params(L, d, V, s)
    print(f"{name:14s} {P / 1e6:7.0f}M {training_state_bytes(P, 'adam') / GB:15.1f} GB "
          f"{transformer_activation_bytes(L, d, a, s, b) / GB:14.1f} GB  ({b} x {s})")

GPT-2 small at batch 8 needs about 2 GB of state and more than 8 GB of activations per step: it is
activation-bound, and either a smaller micro-batch with gradient accumulation (Section 4.1) or
activation checkpointing (Section 4.2) is the natural remedy. The 7-billion-parameter model needs
about 100 GB of state at a batch of *one* sequence: no amount of activation savings helps, and the state
itself must be sharded across GPUs (FSDP/ZeRO, Section 7) or stored in fewer bytes (8-bit optimizers,
Section 4.3). Being able to make this diagnosis on paper, before launching anything, is the practical
value of memory arithmetic.

> **Key idea.** Training memory = 16 bytes × parameters (Adam, fp32 or mixed) + batch size × activation
> elements × bytes per element. Know which term dominates before choosing a remedy.

## 3. Mixed precision

### 3.1 float16, bfloat16 and what can go wrong

A float32 number has 1 sign bit, 8 exponent bits and 23 mantissa bits. The exponent sets the
**range** (roughly $10^{-38}$ to $10^{38}$) and the mantissa sets the **precision** (about 7 decimal
digits, machine epsilon $2^{-23} \approx 1.2 \times 10^{-7}$). The two 16-bit formats used in deep
learning split the 16 bits differently:

| format | exponent bits | mantissa bits | largest value | smallest normal | epsilon |
|---|---|---|---|---|---|
| float32 | 8 | 23 | $3.4 \times 10^{38}$ | $1.2 \times 10^{-38}$ | $1.2 \times 10^{-7}$ |
| float16 | 5 | 10 | $65\,504$ | $6.1 \times 10^{-5}$ | $9.8 \times 10^{-4}$ |
| bfloat16 | 8 | 7 | $3.4 \times 10^{38}$ | $1.2 \times 10^{-38}$ | $7.8 \times 10^{-3}$ |

**float16** keeps three extra mantissa bits but has a tiny range: anything above 65 504 overflows to
infinity, and gradients below about $6 \times 10^{-5}$ become subnormal and below $6 \times 10^{-8}$
become exactly zero. Small gradients are the norm late in training and in deep networks, so float16
training *without* countermeasures silently loses gradient information. **bfloat16** keeps the full
float32 exponent range and gives up precision instead: it cannot distinguish $1.0$ from $1.003$ (the
next representable number above $1$ is $1.0078$).
That sounds terrible, but stochastic gradient descent is remarkably tolerant of noise in the
gradient, provided that the *accumulation* of many products inside a matmul happens in float32, which
tensor cores and modern CPU kernels do internally. This is why bfloat16 has become the default on
hardware that supports it (Ampere and newer GPUs, recent CPUs, TPUs): no loss scaling, no overflow
worries, only a modest loss of precision.

The cell below prints the exact constants from `torch.finfo` and shows the two failure modes on
concrete numbers: gradient underflow in float16 and precision loss in bfloat16.

In [ ]:
print(f"{'dtype':15s} {'bits':>4s} {'eps':>10s} {'max':>11s} {'smallest normal':>16s}")
for dt in (torch.float32, torch.float16, torch.bfloat16):
    fi = torch.finfo(dt)
    print(f"{str(dt):15s} {fi.bits:4d} {fi.eps:10.2e} {fi.max:11.3e} {fi.tiny:16.2e}")

grads = torch.tensor([1e-3, 1e-5, 1e-7, 1e-9])
print("\nsmall gradients :", grads.tolist())
print("  as float16    :", grads.half().tolist(), "  <- underflow to zero below ~6e-8, subnormal below 6e-5")
print("  as bfloat16   :", [f"{v:.3e}" for v in grads.bfloat16().float().tolist()], "  <- range preserved, 3 significant digits")

vals = torch.tensor([1.0, 1.0003, 1.003, 1.1, 65504.0, 70000.0])
print("\nvalues          :", vals.tolist())
print("  as float16    :", vals.half().tolist(), "  <- 1.0003 -> 1.0 (spacing near 1 is ~1e-3); 70000 -> inf")
print("  as bfloat16   :", vals.bfloat16().tolist(), "  <- 1.003 -> 1.0 (spacing near 1 is ~8e-3); 65504 -> 65536, no overflow")

### 3.2 Loss scaling, derived

Mixed-precision training (Micikevicius et al., 2018) keeps a float32 **master copy** of the weights,
runs the forward and backward passes in float16 where that is safe, and applies the update to the
master copy. The remaining problem is gradient underflow. The fix is elegantly simple. Multiply the
loss by a scale factor $S$ before the backward pass. Because backpropagation is linear in the upstream
gradient,

$$
\frac{\partial (S \cdot L)}{\partial \theta} \;=\; S \cdot \frac{\partial L}{\partial \theta},
$$

every gradient in the network is multiplied by $S$, which shifts the whole gradient distribution up
into the representable float16 range; a gradient of $10^{-8}$ becomes $10^{-8} \cdot 2^{16} \approx
6.5 \times 10^{-4}$, comfortably representable. Before the optimizer uses the gradients, they are
divided by $S$ again (in float32, so nothing is lost). The scale must be large enough to lift small
gradients but small enough that the largest gradient does not overflow, and the right value changes
over training. **Dynamic loss scaling** finds it automatically:

1. start with a large $S$ (PyTorch uses $2^{16}$);
2. after each backward pass check the gradients for `inf`/`NaN`; if any is found, *skip* the
   optimizer step and halve $S$;
3. if `growth_interval` consecutive steps (default 2 000) succeed, double $S$.

Because a skipped step costs only one iteration, the scheme spends most of its time at the largest
safe scale. `torch.amp.GradScaler` implements exactly this, and `torch.amp.autocast` implements the
per-operator dtype policy: matmuls and convolutions run in the low-precision type, while reductions
that are numerically delicate (softmax, log-softmax, layer norm, losses, sums) are computed in float32
on CUDA. The canonical training step therefore reads

```python
with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
    loss = loss_fn(model(x), y)          # forward in mixed precision
scaler.scale(loss).backward()            # backward on S * loss
scaler.unscale_(opt)                     # optional: only if you clip gradients before the step
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(opt)                         # unscales, checks for inf/nan, steps or skips
scaler.update()                          # adjusts S
```

With bfloat16 the scaler is unnecessary (the range is that of float32), and the same code works with
`enabled=False` on the scaler. On the CPU, autocast supports bfloat16, so the code path is identical;
whether it is *faster* on a CPU depends on whether the processor has native bfloat16 instructions
(AVX-512 BF16 or AMX); without them the kernels emulate bfloat16 and can be slower than float32.

### 3.3 Benchmark

The function below trains a fresh model for a fixed number of steps under a chosen precision policy
and reports the time per step, the mean loss (to check that training is not silently broken) and the
final loss scale. On a GPU we compare float32, float16 with a scaler and bfloat16 (when the hardware
supports it) on the convolutional model with a larger batch, where tensor cores can shine. On the CPU
we compare float32 with bfloat16 autocast on the MLP, keeping the budget small.

In [ ]:
def train_steps_amp(model_fn, n_steps, amp_dtype=None, batch_size=128, lr=0.05):
    """Train a fresh model for n_steps mini-batches under an autocast policy.
    Returns (seconds per step, mean loss over the timed steps, final loss scale or None)."""
    torch.manual_seed(SEED)
    model = model_fn().to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    use_amp = amp_dtype is not None
    use_scaler = use_amp and amp_dtype == torch.float16 and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    model.train()
    batches = iterate_minibatches(X_train, y_train, batch_size)
    losses = []
    t0 = None
    for step in range(n_steps + 2):                    # two warm-up steps are not timed
        if step == 2:
            sync()
            t0 = time.perf_counter()
        try:
            xb, yb = next(batches)
        except StopIteration:
            batches = iterate_minibatches(X_train, y_train, batch_size)
            xb, yb = next(batches)
        with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        losses.append(loss.detach().float())
    sync()
    per_step = (time.perf_counter() - t0) / n_steps
    return per_step, torch.stack(losses[2:]).mean().item(), (scaler.get_scale() if use_scaler else None)


STEPS_AMP = 30 if FAST_RUN else 150
if device.type == "cuda":
    bench_model, bench_bs = (lambda: SmallCNN(width=64)), 256
    precision_configs = [("float32", None), ("float16 + GradScaler", torch.float16)]
    if torch.cuda.is_bf16_supported():
        precision_configs.append(("bfloat16", torch.bfloat16))
else:
    bench_model, bench_bs = (lambda: MLP(hidden=512, depth=3)), 128
    precision_configs = [("float32", None), ("bfloat16 (CPU autocast)", torch.bfloat16)]

amp_results = {}
for name, dt in precision_configs:
    try:
        per_step, mean_loss, scale = train_steps_amp(bench_model, STEPS_AMP, dt, batch_size=bench_bs)
        amp_results[name] = per_step
        base = amp_results["float32"]
        print(f"{name:26s} {per_step * 1e3:8.2f} ms/step | speed-up vs fp32 {base / per_step:4.2f}x | "
              f"mean loss {mean_loss:.3f}" + (f" | final loss scale {scale:.0f}" if scale is not None else ""))
    except Exception as e:
        print(f"{name:26s} not supported in this environment ({type(e).__name__}: {str(e)[:100]})")

On a GPU with tensor cores (Volta or newer) the float16 and bfloat16 runs are typically 1.5–3× faster
per step for this convolutional model, and the gain grows with model width and batch size because the
matmuls become a larger fraction of the step. The loss scale printed for float16 is usually still
$65\,536$ after a few dozen steps, meaning that no overflow occurred; on a longer run you would see it
occasionally halve and then grow back. The mean losses of all three runs agree to within the noise of
different rounding, which is the sanity check we care about.

On a CPU without native bfloat16 support the bfloat16 run may well be *slower* than float32. That is
not a bug in autocast: the format is being emulated with conversions around float32 kernels, and the
only reason to use it there is to reduce memory. The lesson generalizes: **a precision policy is a
hardware decision**, and the benchmark, not the folklore, decides.

### 3.4 Numerical pitfalls and the autocast policy

Autocast is a *per-operator* policy, and it pays to know which operators it protects. Under CUDA
autocast, `matmul`, `conv`, `linear` and friends run in the low-precision type; `softmax`,
`log_softmax`, `layer_norm`, `cross_entropy`, `sum`, `exp`, `log`, `pow` and other range-sensitive
operators run in float32 and their outputs are float32. The next cell makes this visible by printing
the output dtypes of a few operators inside an autocast region. (The CPU list of float32-protected
operators is different, so the CPU printout will show fewer promotions; the comment in the code
records the CUDA behaviour.)

In [ ]:
amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16
x = torch.randn(4, 8, device=device)
w = torch.randn(8, 8, device=device, requires_grad=True)
target = torch.zeros(4, dtype=torch.long, device=device)
with torch.amp.autocast(device_type=device.type, dtype=amp_dtype):
    h = x @ w                                  # low precision on both devices
    p = F.softmax(h, dim=1)                    # float32 on CUDA
    ln = F.layer_norm(h, (8,))                 # float32 on CUDA
    ce = F.cross_entropy(h, target)            # float32 on CUDA
    s = h.sum()                                # float32 on CUDA
for name, t in [("matmul", h), ("softmax", p), ("layer_norm", ln), ("cross_entropy", ce), ("sum", s)]:
    print(f"{name:14s} -> {t.dtype}")

Three practical rules follow from this behaviour.

- **Never call `model.half()` for training.** That converts the parameters themselves and disables
  the float32 master copy; use autocast, which leaves parameters in float32 and casts on the fly.
- **Write custom numerics in float32.** If you implement your own softmax, log-sum-exp, normalization
  or attention scores with explicit tensor ops, wrap them in
  `with torch.amp.autocast(device_type, enabled=False):` and cast the inputs with `.float()`; autocast
  cannot know that your hand-written `exp(x).sum()` is dangerous.
- **Prefer bfloat16 when the hardware supports it**, keep float16 plus `GradScaler` for older GPUs,
  and check `torch.cuda.is_bf16_supported()` at start-up rather than hard-coding the dtype.

> **Common pitfall.** Computing `loss.backward()` on a float16 loss without a scaler on a deep
> network trains apparently normally for a while and then plateaus: the small gradients of the early
> layers have silently underflowed to zero. Checking `scaler.get_scale()` (should be large and stable)
> and the fraction of skipped steps is part of the health monitoring of any float16 run.

## 4. Fitting bigger models

### 4.1 Gradient accumulation

Suppose the batch size we *want* for optimization reasons is $B = 128$, but only a micro-batch of
$b = 32$ fits in memory. Gradient accumulation recovers the large-batch gradient exactly from $K = B/b$
micro-batches. The mean loss over the big batch is

$$
L_B = \frac{1}{B}\sum_{i=1}^{B} \ell_i
    = \frac{1}{K}\sum_{k=1}^{K} \underbrace{\frac{1}{b}\sum_{i \in \text{micro-batch } k} \ell_i}_{L_{b,k}} ,
$$

so the gradient of the big-batch loss is the *average* of the micro-batch gradients:
$\nabla L_B = \frac{1}{K}\sum_k \nabla L_{b,k}$. Since `loss.backward()` *adds* into `.grad`, we
obtain this average by calling `backward()` on $L_{b,k} / K$ for each micro-batch without zeroing the
gradients in between, and stepping the optimizer once after $K$ micro-batches. Three details are
easy to get wrong:

1. **Divide each micro-batch loss by $K$.** Forgetting this multiplies the gradient by $K$, which for
   SGD is equivalent to multiplying the learning rate by $K$ (for Adam it is nearly harmless because
   Adam normalizes gradient magnitudes, which is why the bug often goes unnoticed until someone
   switches optimizer).
2. **Step the learning-rate scheduler once per optimizer step**, not once per micro-batch; otherwise a
   schedule planned for $T$ updates finishes after $T / K$ updates.
3. **BatchNorm sees the micro-batch, not the big batch.** Its statistics and its normalization are
   computed over $b$ samples, so the equivalence above is exact only for models without batch
   statistics (or when using GroupNorm/LayerNorm); Exercise 2 examines this.

We first verify the equivalence numerically on the MLP (which has no BatchNorm), including the
"forgot to divide" bug.

In [ ]:
def grads_vector(model):
    return torch.cat([p.grad.flatten() for p in model.parameters()])


torch.manual_seed(SEED)
mlp = MLP().to(device)
xb, yb = X_train[:64], y_train[:64]
K, b = 4, 16

mlp.zero_grad(set_to_none=True)                              # (a) one big batch of 64
F.cross_entropy(mlp(xb), yb).backward()
g_big = grads_vector(mlp).clone()

mlp.zero_grad(set_to_none=True)                              # (b) 4 micro-batches of 16, loss / K
for k in range(K):
    (F.cross_entropy(mlp(xb[k * b:(k + 1) * b]), yb[k * b:(k + 1) * b]) / K).backward()
g_acc = grads_vector(mlp).clone()

mlp.zero_grad(set_to_none=True)                              # (c) the bug: forgetting the 1/K
for k in range(K):
    F.cross_entropy(mlp(xb[k * b:(k + 1) * b]), yb[k * b:(k + 1) * b]).backward()
g_bug = grads_vector(mlp).clone()

print(f"|g_big|                = {g_big.norm():.6f}")
print(f"max |g_acc - g_big|    = {(g_acc - g_big).abs().max():.2e}   (accumulation is exact up to rounding)")
print(f"|g_bug| / |g_big|      = {g_bug.norm() / g_big.norm():.4f}   (without the 1/K the gradient is K times too large)")

The accumulated gradient matches the big-batch gradient to floating-point precision, and the buggy
version is exactly $K$ times larger. Now the full training loop with a scheduler. To make the point
that the two procedures are *the same algorithm*, we run one epoch with a genuine batch of 128 and one
epoch with $4 \times 32$ under a one-cycle schedule, using the same seed for shuffling so that the
micro-batches are exactly the slices of the big batches; the losses, accuracies and final learning
rates should coincide.

In [ ]:
def train_with_accumulation(model, opt, X, y, micro_batch, accum_steps, scheduler=None, max_updates=None):
    """One epoch with gradient accumulation: optimizer and scheduler step once per accum_steps micro-batches."""
    model.train()
    opt.zero_grad(set_to_none=True)
    loss_sum = torch.zeros((), device=X.device)
    n_seen, n_micro, n_updates = 0, 0, 0
    for xb, yb in iterate_minibatches(X, y, micro_batch):
        loss = F.cross_entropy(model(xb), yb) / accum_steps   # so that the K accumulated grads average
        loss.backward()                                       # accumulates into .grad
        loss_sum += loss.detach() * accum_steps * xb.shape[0]
        n_seen += xb.shape[0]
        n_micro += 1
        if n_micro % accum_steps == 0:
            opt.step()
            opt.zero_grad(set_to_none=True)
            if scheduler is not None:
                scheduler.step()                              # once per *update*, not per micro-batch
            n_updates += 1
            if max_updates is not None and n_updates >= max_updates:
                break
    opt.zero_grad(set_to_none=True)                           # drop any incomplete accumulation
    return loss_sum.item() / n_seen, n_updates


def run_accumulation_experiment(micro_batch, accum_steps, epochs, lr=0.05):
    torch.manual_seed(SEED)
    model = MLP().to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    updates_per_epoch = N_TRAIN // (micro_batch * accum_steps)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=updates_per_epoch * epochs)
    for ep in range(epochs):
        torch.manual_seed(SEED + 100 + ep)                    # identical shuffles in both runs
        tr, n_up = train_with_accumulation(model, opt, X_train, y_train, micro_batch, accum_steps, sched, updates_per_epoch)
    vl, va = evaluate(model, X_val, y_val)
    return tr, vl, va, n_up, sched.get_last_lr()[0]


EPOCHS_ACC = 1 if FAST_RUN else 3
for label, mb, K in [("batch 128, no accumulation", 128, 1), ("micro-batch 32 x 4 accumulated", 32, 4)]:
    t0 = time.perf_counter()
    tr, vl, va, n_up, last_lr = run_accumulation_experiment(mb, K, EPOCHS_ACC)
    print(f"{label:32s} train loss {tr:.4f} | val loss {vl:.4f} | val acc {va:.4f} | updates/epoch {n_up} | "
          f"final lr {last_lr:.2e} | {time.perf_counter() - t0:.1f}s")

The two rows agree to the printed precision (bitwise equality is not expected: summing 128 losses at
once and summing four means of 32 round differently, and GPU kernels add their own nondeterminism),
confirming that the accumulated run performs the same sequence of updates with the same schedule. The accumulated run is somewhat slower because four small forward/backward passes have
more per-call overhead than one large one; that is the price of fitting into memory.

**Effective batch size and learning rate.** Gradient accumulation, like data parallelism in
Section 7, changes the effective batch size. For SGD the empirical **linear scaling rule** (Goyal et
al., 2017) says that when the batch is multiplied by $K$, the learning rate should be multiplied by
$K$ as well, with a warm-up of a few epochs to survive the large early steps; the intuition is that
$K$ steps with gradient $\nabla L_b$ and rate $\eta$ move about as far as one step with the averaged
gradient and rate $K\eta$. For Adam-style optimizers the rule is weaker and a $\sqrt{K}$ scaling is a
common starting point. Both rules break down at very large batches, where the gradient noise becomes
too small to help exploration; beyond that "critical batch size" more data per step buys nothing.

### 4.2 Activation checkpointing

Activation memory scales with depth: every layer's inputs are kept from the forward pass until the
backward pass reaches them. **Activation checkpointing** (also called gradient checkpointing or
rematerialization; Chen et al., 2016) trades compute for memory: during the forward pass we store only
the inputs of selected *segments* of the network and discard everything inside; during the backward
pass, when a segment's gradient is needed, we re-run its forward from the stored input to rebuild the
intermediate activations, use them, and discard them again. If a network of $L$ layers is split into
$\sqrt{L}$ segments of $\sqrt{L}$ layers, peak activation memory drops from $O(L)$ to $O(\sqrt{L})$ at
the cost of one extra forward pass (roughly a third more compute per step, since the backward pass
costs about twice the forward).

PyTorch exposes this as `torch.utils.checkpoint.checkpoint(fn, *inputs, use_reentrant=False)`: during
the forward pass the intermediate tensors of `fn` are not kept (the non-reentrant implementation installs
saved-tensor hooks that discard them and remember only how to recompute them), and during the backward
pass `fn` is re-run from its stored inputs to rebuild them on demand. `use_reentrant=False` selects this
modern implementation, which composes correctly with everything else (nested checkpoints,
`torch.compile`, tensors that do not require grad) and should always be used; the older reentrant
variant ran `fn` under no-grad and re-executed it as a whole during backward.

We build a deep MLP whose residual-free blocks can optionally be checkpointed, and measure the
activation memory autograd keeps, using the counter from Section 2, together with the time of a
forward-plus-backward pass. On a GPU we additionally report the true peak memory.

In [ ]:
from torch.utils.checkpoint import checkpoint


class DeepMLP(nn.Module):
    """inp -> n_blocks x [Linear, GELU, Linear, GELU] -> out, with optional per-block checkpointing."""

    def __init__(self, d=512, n_blocks=8, use_checkpoint=False):
        super().__init__()
        self.inp = nn.Linear(28 * 28, d)
        self.blocks = nn.ModuleList([nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, d), nn.GELU())
                                     for _ in range(n_blocks)])
        self.out = nn.Linear(d, 10)
        self.use_checkpoint = use_checkpoint

    def forward(self, x):
        h = self.inp(x.flatten(1))
        for blk in self.blocks:
            if self.use_checkpoint and torch.is_grad_enabled():
                h = checkpoint(blk, h, use_reentrant=False)
            else:
                h = blk(h)
        return self.out(h)


D_CKPT, N_BLOCKS, B_CKPT = (512 if FAST_RUN else 1024), 8, 256


def measure_checkpointing(use_checkpoint):
    torch.manual_seed(SEED)
    model = DeepMLP(D_CKPT, N_BLOCKS, use_checkpoint).to(device)
    xb, yb = X_train[:B_CKPT].clone(), y_train[:B_CKPT].clone()   # fresh storage, so the counter does not see the whole dataset tensor

    def fwd_bwd():
        loss = F.cross_entropy(model(xb), yb)
        model.zero_grad(set_to_none=True)
        loss.backward()

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        base = torch.cuda.memory_allocated()
    with SavedTensorCounter(model) as counter:
        fwd_bwd()
    peak = (torch.cuda.max_memory_allocated() - base) if device.type == "cuda" else None
    return counter.bytes, peak, time_fn(fwd_bwd, n_warmup=1, n_repeat=3)


boundary_bytes = N_BLOCKS * B_CKPT * D_CKPT * 4          # block inputs kept by the checkpoint machinery
for flag in (False, True):
    saved, peak, t = measure_checkpointing(flag)
    total = saved + (boundary_bytes if flag else 0)
    line = (f"checkpoint={str(flag):5s} | saved by autograd {saved / MB:7.1f} MB"
            + (f" + segment inputs {boundary_bytes / MB:6.1f} MB = {total / MB:7.1f} MB" if flag else f"{'':36s}")
            + f" | fwd+bwd {t * 1e3:7.1f} ms")
    if peak is not None:
        line += f" | GPU peak {peak / MB:7.1f} MB"
    print(line)

Without checkpointing, autograd keeps four activation tensors of shape $B \times d$ per block
(the inputs of both linear layers and of both GELUs); with checkpointing it keeps only what lies
outside the checkpointed segments (the input image batch and the final block output) plus the one
segment input per block that the checkpoint machinery stores itself: about $8 + 2$ tensors of size
$B \times d$ instead of $4 \cdot 8 + 1$, a reduction by more than a factor of three for this
architecture (and by a factor that grows with the number of layers per segment). The
forward-plus-backward time rises by about 25–35 %, the cost of the recomputed forward. On a GPU the
peak column shows the same reduction net of the fixed cost of parameters and gradients.

In Transformers the same technique is applied per block (Hugging Face models expose it as
`gradient_checkpointing_enable()`), and it is what makes it possible to fine-tune models whose
activations would otherwise not fit, at the price of the extra forward pass. A refinement called
**selective checkpointing** recomputes only the cheap-but-large tensors (attention scores, GELU
inputs) and stores the expensive ones (matmul outputs); Korthikanti et al. (2022) show that this
recovers most of the memory saving for a few percent of extra compute.

### 4.3 8-bit optimizers, quantized training and `channels_last`

**8-bit optimizers.** Section 2 showed that Adam's two moment buffers are half of the 16 bytes per
parameter. Dettmers et al. (2022) store them in 8 bits using *block-wise dynamic quantization*: each
block of 2 048 values gets its own scale, and the values are mapped to an 8-bit code book that is
dense near zero (where most moments live). The moments are dequantized to float32 inside the update
and re-quantized afterwards, so the optimizer's arithmetic is unchanged; the memory drops from 8 to 2
bytes per parameter with no measurable loss in final quality. The `bitsandbytes` library provides
these optimizers (`bnb.optim.Adam8bit`) as drop-in replacements; we do not depend on it in this
course, but the technique is the standard way to fine-tune large models on a single GPU together with
LoRA (Notebook 10 · NLP with Pretrained Transformers (Hugging Face)).

**Quantized training.** Going further, FP8 training (on Hopper-class GPUs) runs the matmuls with 8-bit
floating-point inputs and per-tensor scaling factors, keeping master weights and accumulations in
higher precision; and QLoRA (Dettmers et al., 2023) freezes a base model in 4-bit weights and trains
only low-rank adapters in bfloat16. Both follow the same principle as mixed precision: put the bulk
storage and the bulk arithmetic in fewer bits, keep the numerically sensitive parts in float32.

**Memory format.** Finally, a memory-layout trick for convolutional networks. PyTorch stores a
$(N, C, H, W)$ tensor with the width index varying fastest by default, so that the $H \times W$ values
of one channel are contiguous. cuDNN tensor-core kernels and oneDNN CPU kernels prefer the
**channels-last** layout $(N, H, W, C)$ in memory, in which the $C$ values of one pixel are adjacent; PyTorch supports this as a *memory format* that leaves the
logical shape and all indexing unchanged and only changes the strides. Converting the model and the
input with `.to(memory_format=torch.channels_last)` is usually enough, and under float16 autocast on a
GPU it commonly buys 10–30 % on convolution-heavy models. The next cell shows what the conversion does
to the strides and benchmarks it where that is meaningful.

In [ ]:
x = torch.randn(8, 3, 32, 32)
xc = x.to(memory_format=torch.channels_last)
print("NCHW strides:", x.stride(), "| channels_last contiguous?", x.is_contiguous(memory_format=torch.channels_last))
print("NHWC strides:", xc.stride(), "| shape unchanged:", tuple(xc.shape),
      "| channels_last contiguous?", xc.is_contiguous(memory_format=torch.channels_last))

if device.type == "cuda":
    torch.manual_seed(SEED)
    cnn64 = SmallCNN(width=64).to(device).eval()
    cnn64_cl = copy.deepcopy(cnn64).to(memory_format=torch.channels_last)
    xb = X_val[:256]
    xb_cl = xb.contiguous(memory_format=torch.channels_last)

    def fwd(m, inp):
        with torch.inference_mode(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            return m(inp)

    t_nchw = time_fn(lambda: fwd(cnn64, xb), n_warmup=5, n_repeat=20)
    t_nhwc = time_fn(lambda: fwd(cnn64_cl, xb_cl), n_warmup=5, n_repeat=20)
    print(f"fp16 forward, batch 256: NCHW {t_nchw * 1e3:.2f} ms | channels_last {t_nhwc * 1e3:.2f} ms "
          f"| speed-up {t_nchw / t_nhwc:.2f}x")
else:
    print("No GPU: the channels_last benchmark needs tensor-core convolution kernels to show a difference. "
          "On a CPU the conversion works identically (see the strides above) and oneDNN may also benefit, "
          "but the effect on a 28x28 model is within timing noise.")

The strides tell the story: in NCHW the last stride is 1 along the width, in channels-last it is 1
along the channel axis, while the shape stays $(8, 3, 32, 32)$. Everything else in the code, from
indexing to `nn.Conv2d`, works unchanged; only the kernels selected underneath differ.

> **Key idea.** Three orthogonal levers reduce memory: fewer bytes per element (mixed precision, 8-bit
> states), fewer elements kept (checkpointing, smaller micro-batches with accumulation), and fewer
> elements per device (sharding, Section 7). Each has a compute or communication price, and the memory
> arithmetic of Section 2 tells you which lever addresses the term that is actually too large.

## 5. `torch.compile`, TF32 and matmul precision

### 5.1 What `torch.compile` does

Eager PyTorch executes one operator at a time: Python calls into C++, a kernel is launched, the
result comes back as a tensor, and the next line does the same. For small operators the fixed cost of
this round trip (a few to a few tens of microseconds each) dominates, and separate kernels cannot share
memory traffic: `y = relu(x + b)` reads and writes memory twice, once for the addition and once for the
ReLU. `torch.compile` (Ansel et al., 2024) removes both overheads in three stages:

1. **TorchDynamo** hooks into the CPython frame evaluation, traces the Python bytecode of your
   function into an FX graph of tensor operations, and inserts *guards* (checks on shapes, dtypes and
   Python values) so that the cached graph is only reused when it is still valid. Unsupported Python
   (data-dependent control flow, some third-party calls) causes a **graph break**: the function is
   split into compiled pieces with eager code in between.
2. **AOTAutograd** traces the backward pass as well, so that both directions are compiled.
3. **TorchInductor** generates fused kernels: Triton kernels on GPUs, C++/OpenMP code on CPUs. Elementwise
   chains are fused into one kernel, reductions are fused with their producers, and matmuls are
   dispatched to the best library or to generated templates.

The result is a function that behaves like the original but executes far fewer, larger kernels. The
price is compilation time (seconds to minutes on the first call and again whenever a guard fails, for
instance on a new input shape unless `dynamic=True` is used), the need for a C++ compiler on the
machine, and occasional unsupported constructs. The demo below compiles the MLP (its forward pass and, through AOTAutograd, its backward
pass; the loss and the optimizer step stay eager) and compares the eager and compiled time of a training step. It is guarded because compilation depends on the
platform: it requires a working C++ toolchain (MSVC on Windows, gcc on Linux) and does not support
every Python version, and when it fails the notebook simply reports why and moves on.

In [ ]:
torch.manual_seed(SEED)
comp_model = MLP(hidden=512, depth=3).to(device)
comp_opt = torch.optim.SGD(comp_model.parameters(), lr=0.01)
xb, yb = X_train[:256], y_train[:256]


def step_fn(m):
    loss = F.cross_entropy(m(xb), yb)
    comp_opt.zero_grad(set_to_none=True)
    loss.backward()
    comp_opt.step()
    return loss


t_eager = time_fn(lambda: step_fn(comp_model), n_warmup=3, n_repeat=10)
print(f"eager training step:    {t_eager * 1e3:8.2f} ms")
try:
    compiled_model = torch.compile(comp_model)
    sync()
    t0 = time.perf_counter()
    step_fn(compiled_model)                              # first call triggers compilation
    sync()
    t_first = time.perf_counter() - t0
    t_compiled = time_fn(lambda: step_fn(compiled_model), n_warmup=3, n_repeat=10)
    print(f"first compiled call:    {t_first:8.2f} s  (compilation)")
    print(f"compiled training step: {t_compiled * 1e3:8.2f} ms  | speed-up {t_eager / t_compiled:.2f}x")
    if t_compiled < t_eager:
        print(f"break-even after ~{t_first / (t_eager - t_compiled):,.0f} training steps")
    else:
        print("no per-step gain on this model/device: the step is too small for fusion to matter here")
except Exception as e:
    print(f"torch.compile is not usable here ({type(e).__name__}): {str(e)[:250]}")
    print("Typical causes: no C++ compiler on the machine, an unsupported Python version, or a platform "
          "without Inductor support. Training continues in eager mode with identical results.")

Interpretation depends on what you got. On a GPU, a small model such as this one is *launch-bound* in
eager mode, and fusion typically yields a 1.3–2× faster step; on large models where matmuls dominate,
the gain shrinks toward the 5–15 % that fusing the elementwise operations around them can offer. On a
CPU the compiled step can be faster (fused C++ loops beat many small kernels) or about the same. In all
cases note the *break-even* number: with a compilation cost of tens of seconds and a saving of a
fraction of a millisecond per step, compilation pays off only for runs of many thousands of steps,
which is exactly the regime of real training and exactly not the regime of a notebook experiment.

Common failure modes to recognize:

- **Recompilation storms.** Every new input shape triggers a recompile (up to a cache limit, after
  which the function falls back to eager). Pad variable-length inputs to a few bucket sizes, or pass
  `dynamic=True`.
- **Graph breaks** from `print`, `.item()`, Python-side logging or data-dependent `if` statements
  inside the compiled region. `torch._dynamo.explain(fn)(*args)` lists them.
- **Numerical differences.** Fused kernels reorder floating-point operations (a fused reduction sums
  in a different order than the eager kernel), so agreement with eager to about $10^{-5}$ relative is
  expected and bitwise equality is not.
- **`mode="reduce-overhead"`** uses CUDA graphs to eliminate launch overhead entirely, but requires
  static shapes and static input addresses; `mode="max-autotune"` benchmarks matmul templates at
  compile time and takes much longer to compile.

### 5.2 TF32 and `set_float32_matmul_precision`

**TF32** is a tensor-core format on Ampere and newer NVIDIA GPUs that keeps the 8-bit exponent of
float32 but rounds the mantissa to 10 bits *inside* the multiply, accumulating in float32. It requires
no code changes and no scaler, and it makes float32 matmuls and convolutions several times faster at
the price of a relative precision around $10^{-3}$ per product, which is harmless for neural network
training and dangerous for, say, solving linear systems. Convolutions use TF32 by default in PyTorch;
matmuls do not, and `torch.set_float32_matmul_precision("high")` enables it for them (`"highest"`
restores exact float32, `"medium"` additionally allows bfloat16 internally). The cell below measures
the effect on a large matmul when a suitable GPU is present.

In [ ]:
if device.type == "cuda":
    cap = torch.cuda.get_device_capability()
    n = 4096
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    torch.set_float32_matmul_precision("highest")
    ref = a @ b
    t_highest = time_fn(lambda: a @ b, n_warmup=3, n_repeat=10)
    torch.set_float32_matmul_precision("high")
    out = a @ b
    t_high = time_fn(lambda: a @ b, n_warmup=3, n_repeat=10)
    rel_err = ((out - ref).norm() / ref.norm()).item()
    torch.set_float32_matmul_precision("highest")
    print(f"compute capability {cap[0]}.{cap[1]} | {n}x{n} matmul: highest {t_highest * 1e3:.2f} ms, "
          f"high (TF32) {t_high * 1e3:.2f} ms | speed-up {t_highest / t_high:.2f}x | relative error {rel_err:.1e}")
    if cap[0] < 8:
        print("This GPU predates Ampere, so TF32 is not available and both settings run the same kernel.")
else:
    print("TF32 is a GPU tensor-core format (compute capability >= 8.0). On the CPU "
          "torch.set_float32_matmul_precision has no effect, so there is nothing to measure here.")

On an Ampere or newer GPU you should see a 2–4× faster matmul and a relative error of order $10^{-4}$
to $10^{-3}$, which is far below the noise of stochastic training. The general recommendation is to
call `torch.set_float32_matmul_precision("high")` at the top of any training script that runs in
float32 on such hardware, and to leave it at `"highest"` in numerical code that is not training.

> **Going deeper.** `torch.compile` and TF32 compose with everything in this notebook: a compiled model
> can run under autocast, be wrapped in DDP, and use checkpointing (with `use_reentrant=False`). The
> usual order of application in a production script is: mixed precision first (largest, safest gain),
> then `channels_last` for CNNs, then `torch.compile` once the code is stable, and only then exotic
> options such as CUDA graphs.

## 6. Hyperparameter optimization with Optuna

### 6.1 From random search to model-based search with pruning

Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization ended
with random search and the argument, due to Bergstra and Bengio (2012), that random
search beats grid search because only a few hyperparameters matter and random points cover each of
them densely. Two refinements turn random search into a practical tool for expensive models:

- **Model-based sampling.** After a few random trials, fit a cheap model of "which regions of the
  search space gave good results" and sample the next trial from the promising regions. Optuna's
  default sampler, **TPE** (Tree-structured Parzen Estimator; Bergstra et al., 2011), models the
  density of hyperparameters among the *good* trials, $\ell(x)$, and among the *bad* ones, $g(x)$,
  with kernel density estimates, and proposes candidates that maximize $\ell(x)/g(x)$, which is
  monotone in the expected improvement. It works for mixed continuous, integer and categorical spaces
  with only a few dozen trials.
- **Pruning.** Most bad configurations are recognizably bad early. If a trial reports its validation
  score after every epoch (or every few hundred steps), a pruner can stop trials whose intermediate
  score is below the median of the trials that reached the same step (**MedianPruner**), or below a
  more aggressive quantile. Pruning typically saves half or more of the compute of a study.

Optuna's programming model is *define-by-run*: the objective function receives a `trial` object and
asks it for hyperparameters (`trial.suggest_float("lr", 1e-4, 1e-1, log=True)`), which means the
search space can be built with ordinary Python, including conditional parameters (only ask for
`momentum` if the optimizer is SGD). Inside the objective, `trial.report(value, step)` records an
intermediate value and `trial.should_prune()` asks the pruner whether to stop; raising
`optuna.TrialPruned()` ends the trial cleanly and records it as pruned rather than failed.

### 6.2 The study

We search over the learning rate (log-uniform, the single most important hyperparameter), the hidden
width, the dropout rate, the weight decay (log-uniform) and the choice of optimizer, on the MLP with a
small training subset so that each trial takes about a second. Under `FAST_RUN` we run eight trials of
one epoch with three intermediate reports per epoch; the full setting runs forty trials of four
epochs, still only a few minutes on a GPU. The TPE sampler is seeded so that the study is reproducible.

In [ ]:
N_OPT_TRAIN = 4_000 if FAST_RUN else 20_000
X_opt, y_opt = X_train[:N_OPT_TRAIN], y_train[:N_OPT_TRAIN]
X_opt_val, y_opt_val = X_val[:2_000], y_val[:2_000]
N_TRIALS = 8 if FAST_RUN else 40
EPOCHS_PER_TRIAL = 1 if FAST_RUN else 4
REPORTS_PER_EPOCH = 3
STEPS_PER_EPOCH = math.ceil(N_OPT_TRAIN / 128)


def objective(trial):
    """Train an MLP with trial-suggested hyperparameters; return validation accuracy (to maximize)."""
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    hidden = trial.suggest_categorical("hidden", [64, 128, 256, 512])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    optimizer = trial.suggest_categorical("optimizer", ["sgd", "adamw"])

    torch.manual_seed(SEED)
    model = MLP(hidden=hidden, depth=2, dropout=dropout).to(device)
    if optimizer == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    report_every = max(1, STEPS_PER_EPOCH // REPORTS_PER_EPOCH)
    step = 0
    for epoch in range(EPOCHS_PER_TRIAL):
        model.train()
        for xb, yb in iterate_minibatches(X_opt, y_opt, 128):
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            step += 1
            if step % report_every == 0:
                _, acc = evaluate(model, X_opt_val, y_opt_val)
                trial.report(acc, step)                     # intermediate value for the pruner
                if trial.should_prune():
                    raise optuna.TrialPruned()
                model.train()
    _, acc = evaluate(model, X_opt_val, y_opt_val)
    return acc


if HAVE_OPTUNA:
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=0),
        study_name="fashion_mlp",
    )
    t0 = time.perf_counter()
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    from optuna.trial import TrialState
    completed = [t for t in study.trials if t.state == TrialState.COMPLETE]
    pruned = [t for t in study.trials if t.state == TrialState.PRUNED]
    print(f"{len(study.trials)} trials in {time.perf_counter() - t0:.1f}s: {len(completed)} completed, {len(pruned)} pruned")
    print(f"best value {study.best_value:.4f} with params {study.best_params}")
    print(f"\n{'#':>3s} {'state':9s} {'value':>7s}  params")
    for t in study.trials[:12]:
        val = f"{t.value:.4f}" if t.value is not None else "  -   "
        params = {k: (f"{v:.2e}" if isinstance(v, float) and v < 0.01 else (round(v, 3) if isinstance(v, float) else v))
                  for k, v in t.params.items()}
        print(f"{t.number:3d} {t.state.name:9s} {val:>7s}  {params}")
else:
    print("optuna is not installed; the study is skipped. See environment.yml.")

The printout shows the anatomy of a study: the first three trials are never pruned (`n_startup_trials`),
after which trials whose intermediate accuracy falls below the running median at the same step are
stopped early, usually the ones with a tiny learning rate under SGD, which barely move in one epoch. In
the full run the best configurations reach around 88–89 % validation accuracy, comparable with what
careful manual tuning gives for this model.

### 6.3 Reading a study: history, importance, parallel coordinates

A study is only useful if we learn something from it beyond the single best point; three plots do most
of the work, and we build them with matplotlib directly from `study.trials` so that no plotting
back-end is needed.

1. **Optimization history**: the objective of each completed trial against the trial number, with the
   best-so-far curve. A curve that is still rising at the last trial means the study was too short.
2. **Hyperparameter importance**: Optuna's `get_param_importances` fits a random-forest surrogate to
   the (parameters, value) pairs and decomposes its variance per parameter (fANOVA). It tells us which
   parameters to keep tuning and which to fix.
3. **Parallel coordinates**: each trial is a polyline across one vertical axis per parameter, coloured
   by its objective. Bundles of bright lines show which combinations work; it is the fastest way to
   spot interactions such as "AdamW only with small learning rates".

In [ ]:
if HAVE_OPTUNA and len(completed) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    nums = [t.number for t in completed]
    vals = [t.value for t in completed]
    axes[0].scatter(nums, vals, label="completed trial")
    axes[0].plot(nums, np.maximum.accumulate(vals), color="C3", label="best so far")
    for t in pruned:
        last_step = max(t.intermediate_values) if t.intermediate_values else None
        if last_step is not None:
            axes[0].scatter([t.number], [t.intermediate_values[last_step]], marker="x", color="grey",
                            label="pruned (last report)" if t.number == pruned[0].number else None)
    axes[0].set_xlabel("trial"); axes[0].set_ylabel("validation accuracy"); axes[0].set_title("Optimization history")
    axes[0].legend()

    try:
        importances = optuna.importance.get_param_importances(study)
        names, scores = list(importances.keys()), list(importances.values())
        axes[1].barh(names[::-1], scores[::-1])
        axes[1].set_xlabel("fANOVA importance"); axes[1].set_title("Hyperparameter importance")
    except Exception as e:
        axes[1].text(0.05, 0.5, f"importance unavailable:\n{type(e).__name__}", transform=axes[1].transAxes)
        axes[1].set_title("Hyperparameter importance")
    plt.tight_layout(); plt.show()
else:
    print("Not enough completed trials to plot (or optuna missing).")

With a handful of trials the importance estimate is noisy, but the learning rate and the optimizer
choice usually dominate, followed by width; dropout and weight decay hardly matter in a one-epoch
budget, which is itself a useful finding (regularization only matters once the model can overfit,
which requires more epochs). The parallel-coordinates plot below makes the interactions visible. We
normalize every parameter to $[0, 1]$ (log-scale for the learning rate and the weight decay, category
index for the categorical ones) and colour each polyline by validation accuracy.

In [ ]:
if HAVE_OPTUNA and len(completed) >= 2:
    param_names = ["lr", "hidden", "dropout", "weight_decay", "optimizer"]

    def normalized_column(name, values):
        if name in ("lr", "weight_decay"):
            v = np.log10(np.array(values, dtype=float))
        elif name == "optimizer":
            cats = sorted(set(values))
            v = np.array([cats.index(x) for x in values], dtype=float)
        else:
            v = np.array(values, dtype=float)
        lo, hi = v.min(), v.max()
        return (v - lo) / (hi - lo) if hi > lo else np.full_like(v, 0.5)

    columns = np.stack([normalized_column(p, [t.params[p] for t in completed]) for p in param_names], axis=1)
    scores = np.array([t.value for t in completed])
    norm = plt.Normalize(scores.min(), scores.max())
    cmap = plt.get_cmap("viridis")

    fig, ax = plt.subplots(figsize=(9, 4))
    order = np.argsort(scores)                              # draw the best trials last (on top)
    for i in order:
        ax.plot(range(len(param_names)), columns[i], color=cmap(norm(scores[i])), alpha=0.8, lw=1.5)
    ax.set_xticks(range(len(param_names)))
    ax.set_xticklabels(param_names)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["low", "high"])
    ax.set_title("Parallel coordinates of completed trials (colour = validation accuracy)")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig.colorbar(sm, ax=ax, label="validation accuracy")
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots()
    for opt_name, marker in [("sgd", "o"), ("adamw", "s")]:
        pts = [(t.params["lr"], t.value) for t in completed if t.params["optimizer"] == opt_name]
        if pts:
            ax.scatter([p[0] for p in pts], [p[1] for p in pts], marker=marker, label=opt_name)
    ax.set_xscale("log"); ax.set_xlabel("learning rate"); ax.set_ylabel("validation accuracy")
    ax.set_title("Slice plot: learning rate vs accuracy by optimizer"); ax.legend()
    plt.tight_layout(); plt.show()

The slice plot is the classic picture of a learning-rate search: accuracy rises with the learning rate
up to an optimum and then collapses, and the optimum sits at a different place for SGD (around
$10^{-1}$ with momentum) and for AdamW (around $10^{-3}$ to $10^{-2}$). A search space that spans both
regimes is therefore best expressed with a *conditional* learning-rate range per optimizer, which the
define-by-run API makes trivial (Exercise 4 does this).

Practical guidance for real studies: (i) start with a coarse, wide, log-scaled space and a few dozen
trials, then narrow; (ii) prune on a metric that is comparable across trials at the same step, and
never on the training loss of models with different regularization; (iii) store the study in a
database (`storage="sqlite:///study.db"`) so that it survives a crash and can be resumed or run from
several processes at once; (iv) treat the best trial as a hypothesis and re-train it with several seeds
before believing it.

> **Common pitfall.** Tuning on the test set. The study's objective must be computed on a validation
> split; the test set is looked at once, at the very end, with the chosen configuration.

## 7. Distributed training concepts and a DDP script

### 7.1 The forms of parallelism

When one device is too slow or too small, we use several. There are four ways to split the work, and
real systems combine them:

- **Data parallelism.** Every device holds a full copy of the model and processes a different slice of
  the mini-batch; after the backward pass the gradients are *averaged* across devices so that all copies
  perform the same update and stay identical. The effective batch size is the per-device batch times
  the number of devices, which brings the learning-rate scaling rules of Section 4.1 into play. This is
  the workhorse for everything that fits on one device.
- **Tensor (intra-layer) parallelism.** A single large matmul is split across devices along its
  output (or input) dimension; each device computes a slice, and an all-reduce or all-gather
  reassembles the result inside every layer. This needs a very fast interconnect (NVLink within a
  node) and is how 70B-parameter models are spread over 8 GPUs.
- **Pipeline (inter-layer) parallelism.** Consecutive groups of layers live on different devices; a
  mini-batch is split into micro-batches that flow through the stages like an assembly line. The
  "bubble" at the start and end of each step wastes a fraction of the time equal to roughly
  $(\text{stages}-1)/\text{micro-batches}$, so many micro-batches are needed.
- **Sharded data parallelism (FSDP, ZeRO).** Data parallelism whose *training state* is sharded rather
  than replicated. Rajbhandari et al. (2020) describe three stages: ZeRO-1 shards the optimizer state,
  ZeRO-2 also shards the gradients, ZeRO-3 also shards the parameters, which are then all-gathered
  layer by layer just before they are needed and freed afterwards. With $N$ devices and the
  mixed-precision accounting of Section 2 (2-byte parameters and gradients plus 12 bytes of float32
  master weights and Adam moments, which ZeRO counts as optimizer state), the 16 bytes per parameter
  become $4 + 12/N$, $2 + 14/N$ and $16/N$ bytes per device respectively; ZeRO-3 with 64 GPUs trains
  a model 64 times larger than plain data parallelism in the same memory, at the cost of roughly 1.5×
  the communication volume. PyTorch's `FullyShardedDataParallel` implements ZeRO-3.

### 7.2 How DistributedDataParallel works

`torch.nn.parallel.DistributedDataParallel` (DDP; Li et al., 2020) implements data parallelism with
one *process* per device. The mechanics:

1. **Process group.** `torch.distributed.init_process_group(backend)` connects the processes; the
   launcher (`torchrun`) tells each process its `RANK` (global index), `LOCAL_RANK` (index on its
   machine), `WORLD_SIZE`, and the address of rank 0 for the initial rendezvous. The backend is `nccl`
   for GPUs and `gloo` for CPUs.
2. **Broadcast at construction.** Wrapping the model in `DDP(model)` broadcasts rank 0's parameters and
   buffers to every rank, so all replicas start identical.
3. **All-reduce during backward.** DDP registers autograd hooks on every parameter. As gradients become
   ready during `loss.backward()`, they are collected into **buckets** (25 MB by default, in reverse
   order of the parameters, which is the order in which gradients are produced) and each full bucket is
   all-reduced *while the rest of the backward pass is still running*, overlapping communication with
   computation. The all-reduce sums the gradients and DDP divides by the world size, so every rank ends
   up with the average gradient.
4. **Identical updates.** Because every rank has the same parameters and the same averaged gradient,
   the local `optimizer.step()` produces the same result everywhere; no parameter communication is
   needed.

A **ring all-reduce** of $M$ bytes among $P$ devices moves $2(P-1)/P \cdot M \approx 2M$ bytes through
each device's link, independent of $P$: for a 1-billion-parameter model in float16 that is 4 GB per
step, which at 100 GB/s of NVLink bandwidth costs 40 ms; if a step computes for 400 ms the overhead is
10 % even without overlap. The ratio of computation to communication improves with larger per-device
batches (more compute per byte of gradient), which is one reason large models train with large global
batches. Two further details matter in practice: `DistributedSampler` gives every rank a disjoint
slice of the dataset and must be told the epoch (`sampler.set_epoch(epoch)`) so that the shuffle
changes; and `model.no_sync()` suspends the all-reduce for the first $K-1$ micro-batches of a
gradient-accumulation loop.

### 7.3 A complete DDP training script

Distributed training cannot run inside a notebook cell: each rank must be its own process launched by
`torchrun`. The next cell therefore writes a complete, self-contained script to `DATA_DIR`, validates
its syntax, and prints the launch commands. Every distributed-specific line is commented. The script
is not executed here (a single-process notebook has no process group), but it runs as-is on a machine
with two GPUs, and with `--backend gloo` on a CPU-only machine as two processes.

In [ ]:
DDP_SCRIPT = r'''
"""ddp_train.py - minimal DistributedDataParallel training of an MLP on FashionMNIST.

Launch on one machine with two GPUs (or two CPU processes with --backend gloo):

    torchrun --standalone --nproc_per_node=2 ddp_train.py --epochs 2 --data-dir ../data

torchrun starts one process per --nproc_per_node and sets the environment variables
RANK, LOCAL_RANK, WORLD_SIZE, MASTER_ADDR and MASTER_PORT that init_process_group reads.
"""
import argparse
import os
import time

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset
from torchvision import datasets


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data-dir", default="../data")
    p.add_argument("--epochs", type=int, default=2)
    p.add_argument("--batch-size", type=int, default=128, help="per-process batch size")
    p.add_argument("--lr", type=float, default=0.05, help="learning rate for a global batch of --batch-size")
    p.add_argument("--backend", default=None, help="nccl (default when CUDA is available) or gloo (CPU)")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--workers", type=int, default=0)
    return p.parse_args()


class MLP(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, 10))

    def forward(self, x):
        return self.net(x.flatten(1))


def load_split(data_dir, train):
    ds = datasets.FashionMNIST(data_dir, train=train, download=True)
    X = (ds.data.float().div(255.0).unsqueeze(1) - 0.2860) / 0.3530
    return TensorDataset(X, ds.targets.clone())


def setup_distributed(backend):
    if backend is None:
        backend = "nccl" if torch.cuda.is_available() else "gloo"
    dist.init_process_group(backend=backend)            # rendezvous through the torchrun environment variables
    rank, world_size = dist.get_rank(), dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    if backend == "nccl":
        torch.cuda.set_device(local_rank)                # exactly one GPU per process
        device = torch.device("cuda", local_rank)
    else:
        device = torch.device("cpu")
    return rank, world_size, local_rank, device


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = torch.zeros((), device=device)
    total = torch.zeros((), device=device)
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum()
        total += yb.numel()
    dist.all_reduce(correct)                             # every rank evaluated its own shard: sum them
    dist.all_reduce(total)
    return (correct / total).item()


def main():
    args = parse_args()
    rank, world_size, local_rank, device = setup_distributed(args.backend)
    is_main = rank == 0
    torch.manual_seed(args.seed)                         # same initialization on every rank

    train_ds, val_ds = load_split(args.data_dir, True), load_split(args.data_dir, False)
    train_sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True, seed=args.seed)
    val_sampler = DistributedSampler(val_ds, num_replicas=world_size, rank=rank, shuffle=False)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, sampler=train_sampler,
                              num_workers=args.workers, pin_memory=(device.type == "cuda"), drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=1000, sampler=val_sampler, num_workers=args.workers)

    model = MLP().to(device)
    model = DDP(model, device_ids=[local_rank] if device.type == "cuda" else None)   # broadcasts rank 0's weights
    lr = args.lr * world_size                            # linear scaling rule: global batch = batch_size * world_size
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    steps_per_epoch = len(train_loader)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps_per_epoch * args.epochs)
    if is_main:
        print(f"world_size={world_size} backend={dist.get_backend()} device={device} "
              f"global batch={args.batch_size * world_size} lr={lr}")

    for epoch in range(args.epochs):
        train_sampler.set_epoch(epoch)                   # new shuffle each epoch, consistent across ranks
        model.train()
        t0 = time.time()
        loss_sum = torch.zeros((), device=device)
        for xb, yb in train_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()                              # DDP all-reduces (averages) the gradients in here
            opt.step()                                   # identical update on every rank
            sched.step()
            loss_sum += loss.detach()
        dist.all_reduce(loss_sum)                        # aggregate the training loss over ranks
        train_loss = loss_sum.item() / (steps_per_epoch * world_size)
        val_acc = evaluate(model, val_loader, device)
        if is_main:
            print(f"epoch {epoch + 1} | train loss {train_loss:.4f} | val acc {val_acc:.4f} | {time.time() - t0:.1f}s")
            torch.save(model.module.state_dict(), os.path.join(args.data_dir, "ddp_mlp.pt"))   # unwrap .module
        dist.barrier()                                   # keep ranks in step; rank 0 finishes saving first

    dist.destroy_process_group()


if __name__ == "__main__":
    main()
'''.lstrip()

script_path = DATA_DIR / "ddp_train.py"
script_path.write_text(DDP_SCRIPT, encoding="utf-8")
compile(DDP_SCRIPT, str(script_path), "exec")            # syntax check only; nothing is executed
print(f"wrote {script_path} ({len(DDP_SCRIPT.splitlines())} lines, syntax OK)\n")
print("Launch commands (run in a terminal, not in this notebook):")
print(f"  cd {DATA_DIR}")
print("  torchrun --standalone --nproc_per_node=2 ddp_train.py --epochs 2            # two GPUs on this machine")
print("  torchrun --standalone --nproc_per_node=2 ddp_train.py --epochs 1 --backend gloo   # two CPU processes")
print("  torchrun --nnodes=2 --nproc_per_node=8 --rdzv_backend=c10d --rdzv_endpoint=HOST:29400 ddp_train.py   # multi-node")

Reading the script top to bottom, the distributed-specific lines are few: `init_process_group`,
`DistributedSampler` with `set_epoch`, the `DDP(...)` wrapper with `device_ids`, the all-reduces used
to aggregate metrics, the `rank == 0` guard around printing and checkpointing, and
`destroy_process_group`. Everything else is the ordinary training loop from Notebook 05 · The PyTorch
Workflow and Best Practices. Note the
`model.module.state_dict()` when saving: the DDP wrapper adds a `module.` prefix to every key, and
saving the unwrapped module keeps the checkpoint loadable by a non-distributed script.

A few operational facts round out the picture:

- `torchrun --standalone` starts a rendezvous server in-process; on a cluster, `--nnodes`,
  `--rdzv_endpoint` and the scheduler's environment (Slurm, Kubernetes) take over.
- **Learning rate.** The script applies the linear scaling rule for SGD; a warm-up (OneCycleLR's
  rising phase here) is what makes the larger rate survivable. For AdamW, scale less aggressively.
- **Logging and evaluation** should happen on rank 0 only, but *every* rank must execute the same
  collective calls in the same order (all-reduce, barrier), otherwise the others wait forever: a
  `return` on rank 0 before a barrier is the classic hang.
- **Checkpoint resume** must restore the same state on all ranks; loading on rank 0 and letting DDP's
  construction-time broadcast propagate it is the simplest pattern.
- **FSDP** replaces the `DDP(...)` line by `FullyShardedDataParallel(model, auto_wrap_policy=...)`
  and, for models that do not fit on one device at all, materializes parameters directly on their
  shard; the rest of the loop is unchanged, but checkpoints must be gathered with the FSDP state-dict
  utilities.

> **Key idea.** Data parallelism keeps replicas identical by averaging gradients; its cost is one
> all-reduce of the full gradient per step, hidden behind the backward pass by bucketing. Sharding
> (ZeRO/FSDP) removes the replication of the training state and is what lets the memory arithmetic
> of Section 2 scale beyond one device.

## 8. Scaling laws, compute budgets and experiment hygiene

### 8.1 The $6ND$ rule

How much compute does training cost? For a dense model with $N$ parameters that "touch" each token
(every weight matrix participates in one multiply-add per token), the forward pass costs about $2N$
floating-point operations per token: one multiplication and one addition per weight. The backward pass
costs twice that, $4N$, because for each layer it computes both the gradient with respect to the input
(to keep propagating) and the gradient with respect to the weights. Training on $D$ tokens therefore
costs

$$
C \;\approx\; 6\,N\,D \quad \text{FLOPs}.
$$

This ignores the attention-score computation ($\propto$ sequence length, not parameters), embeddings
(a lookup, not a matmul) and the small savings of the first layer (its input needs no gradient), and
it is accurate to within ten to twenty percent for Transformers whose context is shorter than a few
thousand tokens (Kaplan et al., 2020). The cell below checks the two constants empirically with
`torch.utils.flop_counter.FlopCounterMode`, which counts the FLOPs of every matmul and convolution that
actually executes, and then applies the rule to a few well-known models together with a wall-clock
estimate at a realistic **model FLOP utilization** (MFU) of 40 %: real training rarely achieves more
than 35–50 % of the GPU's peak because of memory traffic, communication and small kernels.

In [ ]:
try:
    from torch.utils.flop_counter import FlopCounterMode
    torch.manual_seed(SEED)
    flop_model = MLP(hidden=512, depth=2).to(device)
    B_FLOP = 256
    xb, yb = X_train[:B_FLOP], y_train[:B_FLOP]
    n_weights = sum(p.numel() for p in flop_model.parameters() if p.dim() > 1)   # matrices only; biases add no matmul FLOPs
    with FlopCounterMode(display=False) as fc_fwd:
        with torch.no_grad():
            flop_model(xb)
    fwd_flops = fc_fwd.get_total_flops()
    with FlopCounterMode(display=False) as fc_all:
        F.cross_entropy(flop_model(xb), yb).backward()
    total_flops = fc_all.get_total_flops()
    first_layer = 784 * 512
    print(f"weights N = {n_weights:,} | batch B = {B_FLOP}")
    print(f"forward FLOPs / (2 N B)      = {fwd_flops / (2 * n_weights * B_FLOP):.3f}")
    print(f"forward+backward / (6 N B)   = {total_flops / (6 * n_weights * B_FLOP):.3f}")
    print(f"corrected for the first layer (no input gradient needed): "
          f"{total_flops / (6 * n_weights * B_FLOP - 2 * first_layer * B_FLOP):.3f}")
except Exception as e:
    print("FlopCounterMode unavailable in this build:", type(e).__name__, str(e)[:120])


def training_flops(n_params, n_tokens):
    return 6.0 * n_params * n_tokens


def training_hours(flops, n_gpus, peak_flops_per_gpu=312e12, mfu=0.4):
    """Wall-clock hours at a given peak (A100 bf16: 312 TFLOP/s) and model FLOP utilization."""
    return flops / (n_gpus * peak_flops_per_gpu * mfu) / 3600


print(f"\n{'model':32s} {'N':>7s} {'D (tokens)':>11s} {'FLOPs':>9s} {'A100-hours':>11s} {'days on 256 GPUs':>17s}")
for name, N, D in [("GPT-2 small (124M, WebText ~9B)", 124e6, 9e9), ("GPT-3 (175B, 300B)", 175e9, 300e9),
                   ("Chinchilla (70B, 1.4T)", 70e9, 1.4e12), ("Llama-2 7B (2T)", 7e9, 2e12)]:
    C = training_flops(N, D)
    print(f"{name:32s} {N / 1e9:6.2f}B {D / 1e9:9.0f}B {C:9.1e} {training_hours(C, 1):11,.0f} {training_hours(C, 256) / 24:17.1f}")

The forward count matches $2NB$ exactly (the counter counts one multiply-add as two FLOPs, as the rule
assumes), and the full step comes out below $6NB$ by precisely the first layer's input-gradient term,
which the corrected line shows; for a deep model that term is negligible and the rule is accurate. The
table then puts the numbers in perspective: GPT-2 small is an afternoon on a handful of GPUs, while
the 2022–2023 generation of open models costs hundreds of thousands to millions of A100-hours. Since
GPU-hours have a price,
$C = 6ND$ converts directly into a budget, and the question becomes how to *spend* a given $C$.

### 8.2 Scaling laws and compute-optimal models

Kaplan et al. (2020) observed that the test loss of Transformer language models follows power laws in
each of $N$, $D$ and $C$ over many orders of magnitude when the other quantities are not the
bottleneck, for instance $L(N) \propto N^{-0.076}$ and $L(D) \propto D^{-0.095}$. Hoffmann et al.
(2022), the "Chinchilla" paper, fitted a joint parametric form,

$$
L(N, D) \;=\; E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}},
$$

where $E$ is the irreducible loss of natural text, the second term is the penalty for a model that is
too small, and the third the penalty for too little data. Minimizing $L$ subject to $6ND = C$ gives the
compute-optimal allocation; with the fitted exponents both $N_{\text{opt}}$ and $D_{\text{opt}}$ grow
roughly as $\sqrt{C}$, so the *ratio* $D/N$ stays approximately constant at about **20 tokens per
parameter**. This overturned the earlier practice of training very large models on comparatively few
tokens: GPT-3 (175B parameters, 300B tokens, 1.7 tokens per parameter) and Gopher (280B, 300B tokens)
were, by this criterion, far too large for their data, and Chinchilla (70B, 1.4T tokens), trained with
the same compute as Gopher, outperformed both with a quarter of Gopher's parameters. Later open models train well past the compute-optimal point (Llama-2 7B saw about
300 tokens per parameter) because a *smaller* model with the same loss is cheaper to *serve*; the
optimum shifts once inference cost is included in the objective.

The cell below plots iso-compute curves of the parametric form with illustrative constants of the
same shape and magnitude as the published fits (the exact constants differ between the original fit
and later re-analyses; ours are chosen so that the optimum lies at 20 tokens per parameter), and
prints the compute-optimal size for a few budgets.

In [ ]:
E_IRR, A_COEF, B_COEF, ALPHA, BETA = 1.7, 400.0, 1140.0, 0.35, 0.35      # illustrative constants


def parametric_loss(N, D):
    return E_IRR + A_COEF / N ** ALPHA + B_COEF / D ** BETA


def chinchilla_optimal(compute_flops, tokens_per_param=20.0):
    N = math.sqrt(compute_flops / (6.0 * tokens_per_param))
    return N, tokens_per_param * N


Ns = np.logspace(7.5, 11.5, 400)
plt.figure(figsize=(7, 4))
for C in (1e19, 1e20, 1e21, 1e22, 1e23):
    Ds = C / (6 * Ns)
    L = parametric_loss(Ns, Ds)
    i = int(np.argmin(L))
    plt.plot(Ns, L, label=f"C = {C:.0e} FLOPs")
    plt.plot(Ns[i], L[i], "ko", ms=4)
plt.xscale("log"); plt.ylim(1.7, 3.5)
plt.xlabel("parameters N (tokens D = C / 6N)"); plt.ylabel("loss (parametric fit)")
plt.title("Iso-compute curves: the minimum moves right as the budget grows")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

print(f"{'budget C':>10s} {'N_opt':>9s} {'D_opt':>10s}   (20 tokens per parameter)")
for C in (1e19, 1e20, 1e21, 1e22, 1e23, 1e24):
    N_opt, D_opt = chinchilla_optimal(C)
    print(f"{C:10.0e} {N_opt / 1e9:8.2f}B {D_opt / 1e9:9.0f}B")

Each curve is one compute budget; moving right trades data for parameters at fixed cost. The minima
(black dots) shift to larger models as the budget grows, and the curves are flat near the minimum:
being off by a factor of two in $N$ costs only a few hundredths of a nat of loss, so in practice one
picks a model size that is convenient for serving and near the optimum, then trains it for as many
tokens as the budget allows. To use this in a project: (1) estimate the budget in FLOPs from the
GPU-hours you have and a 40 % MFU; (2) pick $N \approx \sqrt{C / 120}$ or smaller; (3) make sure you
actually have $20N$ or more tokens of data, since data-limited training must fall back to smaller
models or more epochs, whose returns diminish quickly.

### 8.3 Experiment hygiene: configs, logs and ablations

All of the above assumes that the numbers we compare come from experiments that differ in exactly one
thing. That is a discipline, and it is enforced by tooling, not by good intentions. The pattern used by
every serious research code base is the same:

- a single **configuration object** holds every knob (architecture, optimizer, data subset, seed,
  budget); the training function takes only that object; no hyperparameter lives in code or in a
  notebook cell;
- the config is **serialized next to the results** (JSON or YAML) so that a result can always be
  traced to the exact settings that produced it, and so that a run can be repeated by loading the
  file;
- **metrics are logged per epoch** to an append-only file (JSON lines is the simplest format that
  tools can read) together with the environment (library versions, device, code version);
- **ablations change one field at a time** from a fixed baseline, and every configuration is run with
  several seeds so that the comparison is a mean with an error bar rather than a single number.

A dataclass is the lightest possible implementation of this pattern, and `dataclasses.replace` gives
one-line ablations.

In [ ]:
from dataclasses import replace


@dataclass
class TrainConfig:
    """Every knob of a run, serializable to JSON."""
    run_name: str
    model: str = "mlp"
    hidden: int = 256
    depth: int = 2
    lr: float = 0.05
    batch_size: int = 128
    epochs: int = 1 if FAST_RUN else 3
    weight_decay: float = 0.0
    optimizer: str = "sgd"
    seed: int = SEED
    n_train: int = N_TRAIN
    notes: str = ""


RUNS_DIR = DATA_DIR / "runs_nb14"


def run_experiment(cfg):
    """Train according to cfg, writing config.json, metrics.jsonl and environment.json into its run folder."""
    run_dir = RUNS_DIR / cfg.run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "config.json").write_text(json.dumps(asdict(cfg), indent=2), encoding="utf-8")
    random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
    model = (MLP(cfg.hidden, cfg.depth) if cfg.model == "mlp" else SmallCNN()).to(device)
    if cfg.optimizer == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=0.9, weight_decay=cfg.weight_decay)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    Xtr, ytr = X_train[:cfg.n_train], y_train[:cfg.n_train]
    history = []
    with open(run_dir / "metrics.jsonl", "w", encoding="utf-8") as f:
        for epoch in range(cfg.epochs):
            t0 = time.perf_counter()
            tr = train_one_epoch(model, opt, Xtr, ytr, cfg.batch_size)
            vl, va = evaluate(model, X_val, y_val)
            rec = {"epoch": epoch + 1, "train_loss": round(tr, 5), "val_loss": round(vl, 5),
                   "val_acc": round(va, 5), "seconds": round(time.perf_counter() - t0, 2)}
            f.write(json.dumps(rec) + "\n")
            history.append(rec)
    env = {"torch": torch.__version__, "device": str(device), "cuda": torch.version.cuda,
           "finished": time.strftime("%Y-%m-%d %H:%M:%S")}
    (run_dir / "environment.json").write_text(json.dumps(env, indent=2), encoding="utf-8")
    return history


baseline = TrainConfig("baseline", notes="reference configuration")
ablations = [baseline,
             replace(baseline, run_name="lr_x2", lr=0.1),
             replace(baseline, run_name="wide_512", hidden=512),
             replace(baseline, run_name="adamw_1e-3", optimizer="adamw", lr=1e-3)]
for cfg in ablations:
    hist = run_experiment(cfg)
    last = hist[-1]
    print(f"{cfg.run_name:12s} epoch {last['epoch']} | train {last['train_loss']:.4f} | val {last['val_loss']:.4f} "
          f"| acc {last['val_acc']:.4f} | {last['seconds']:.1f}s")

The results are on disk, and the analysis can be written *once* as a function that reads every run
folder, which is how a results table for a paper or a report is produced reproducibly rather than by
copying numbers out of notebook outputs. The cell below reads the folders back and prints the table
sorted by validation accuracy, showing the ablated field for each run relative to the baseline.

In [ ]:
def load_runs(runs_dir):
    rows = []
    for run_dir in sorted(p for p in runs_dir.iterdir() if p.is_dir()):
        cfg_file, met_file = run_dir / "config.json", run_dir / "metrics.jsonl"
        if not (cfg_file.exists() and met_file.exists()):
            continue
        cfg = json.loads(cfg_file.read_text(encoding="utf-8"))
        records = [json.loads(line) for line in met_file.read_text(encoding="utf-8").splitlines() if line.strip()]
        if records:
            rows.append((cfg, records[-1]))
    return rows


base_cfg = asdict(baseline)
rows = load_runs(RUNS_DIR)
rows.sort(key=lambda r: -r[1]["val_acc"])
print(f"{'run':12s} {'val acc':>8s} {'val loss':>9s} {'epochs':>6s}  changed vs baseline")
for cfg, last in rows:
    changed = {k: v for k, v in cfg.items() if k not in ("run_name", "notes") and base_cfg.get(k) != v}
    print(f"{cfg['run_name']:12s} {last['val_acc']:8.4f} {last['val_loss']:9.4f} {cfg['epochs']:6d}  {changed if changed else '(baseline)'}")

Whatever the ranking on your machine, remember that it rests on one seed and one short budget; the
correct next step before drawing any conclusion is to run the top candidates with three seeds and
report mean and standard deviation, and to check that the ranking is stable when the budget is
increased (a learning rate that wins after one epoch often loses after ten). The tooling makes this
cheap: `replace(cfg, seed=s)` in a loop and the same `load_runs` table.

Beyond this minimal pattern, experiment trackers (TensorBoard, which we used in Notebook 05 · The
PyTorch Workflow and Best Practices, or
services such as Weights & Biases and MLflow) add dashboards, comparison views and artifact storage,
and configuration frameworks (Hydra, OmegaConf) add command-line overrides and composition of config
files. They are conveniences on top of the same idea: one config, one run folder, append-only metrics.

> **Key idea.** Compute is a budget ($C = 6ND$), model size is a choice made from that budget
> ($D \approx 20N$), and the only experiments worth running are ones whose configuration and results
> are recorded well enough to be repeated.

## Exercises

### Exercise 1 (warm-up): profile and speed up a slow training loop

The function `slow_epoch` below trains the MLP for one epoch and is deliberately written the way many
first training loops are written. Time it, identify its three inefficiencies (Section 1 discussed all
of them), and write `fast_epoch` that produces the same kind of training but runs several times
faster. Report the speed-up.

In [ ]:
def slow_epoch(model, opt, batch_size=32):
    """A deliberately inefficient epoch: CPU DataLoader with per-sample indexing, a NumPy round trip for
    every batch, a tiny batch size, and a device synchronization on every step."""
    loader = DataLoader(TensorDataset(X_train.cpu(), y_train.cpu()), batch_size=batch_size, shuffle=True)
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb = torch.from_numpy(xb.cpu().numpy().astype(np.float32)).to(device)   # pointless round trip
        yb = yb.to(device)
        loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        total += loss.item()                                                 # sync every step
    return total / len(loader)


torch.manual_seed(SEED)
slow_model = MLP().to(device)
slow_opt = torch.optim.SGD(slow_model.parameters(), lr=0.05, momentum=0.9)
t0 = time.perf_counter()
slow_loss = slow_epoch(slow_model, slow_opt)
sync()
t_slow = time.perf_counter() - t0
print(f"slow_epoch: mean loss {slow_loss:.4f} in {t_slow:.2f}s")

**Solution to Exercise 1**

The three problems are (1) the batch is assembled sample by sample on the CPU by the `DataLoader`
and then copied to the device every step, although the data already fits in device memory;
(2) the NumPy round trip copies the batch twice more for no reason; (3) `loss.item()` forces a
synchronization on every step, and the batch size of 32 makes the per-step overheads (kernel launches,
optimizer update) dominate. The fast version slices the device-resident tensors, uses a batch of 128
(with the learning rate unchanged for simplicity; Section 4.1 discussed scaling it) and accumulates
the loss on the device. It is the `train_one_epoch` helper from Section 1.2.

In [ ]:
def fast_epoch(model, opt, batch_size=128):
    return train_one_epoch(model, opt, X_train, y_train, batch_size)


torch.manual_seed(SEED)
fast_model = MLP().to(device)
fast_opt = torch.optim.SGD(fast_model.parameters(), lr=0.05, momentum=0.9)
t0 = time.perf_counter()
fast_loss = fast_epoch(fast_model, fast_opt)
sync()
t_fast = time.perf_counter() - t0
print(f"fast_epoch: mean loss {fast_loss:.4f} in {t_fast:.2f}s  -> speed-up {t_slow / t_fast:.1f}x")
print(f"validation accuracy: slow {evaluate(slow_model, X_val, y_val)[1]:.3f} | fast {evaluate(fast_model, X_val, y_val)[1]:.3f}")

The speed-up is typically 5–20× depending on the device, and the validation accuracies are similar
(the slow loop performs four times as many, smaller updates, which slightly favours it in a single
epoch). If you profile `slow_epoch` with the tools of Section 1.3, the top rows are not the model at
all but `aten::stack`/`aten::copy_` from collation and copying, which is the signature of a data-bound
loop.

### Exercise 2: gradient accumulation with BatchNorm and a scheduler

Section 4.1 showed that accumulation is exactly equivalent to a large batch for the MLP. Repeat the
experiment with `SmallCNN`, which contains BatchNorm layers: train for one epoch with a batch of 128
and with $4 \times 32$ accumulated micro-batches, using the same shuffling seed and a one-cycle
schedule. (a) Verify that the scheduler performs the same number of steps in both runs. (b) Show that
the two runs are *not* identical and explain why. (c) Replace BatchNorm by GroupNorm (which normalizes
each sample separately) and show that the equivalence is restored.

**Solution to Exercise 2**

BatchNorm computes its mean and variance over the micro-batch of 32, so each micro-batch is normalized
with different statistics than the full batch of 128 would use, and the running statistics used at
evaluation time are updated four times per update instead of once. The forward pass, and hence the
gradient, therefore differs between the two runs even though the optimizer sees the "same" batch.
GroupNorm normalizes over channels within each sample and has no batch dependence, so with it the two
runs coincide. The scheduler count is identical in both cases because `train_with_accumulation` steps
it once per optimizer update.

In [ ]:
class CNNWithNorm(nn.Module):
    """SmallCNN with a selectable normalization: 'bn' (batch statistics) or 'gn' (per-sample groups)."""

    def __init__(self, norm="bn", width=32, n_classes=10):
        super().__init__()
        def make_norm(c):
            return nn.BatchNorm2d(c) if norm == "bn" else nn.GroupNorm(8, c)
        self.features = nn.Sequential(
            nn.Conv2d(1, width, 3, padding=1), make_norm(width), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(width, 2 * width, 3, padding=1), make_norm(2 * width), nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(2 * width * 7 * 7, 128), nn.ReLU(), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


def accumulation_run(norm, micro_batch, accum_steps, epochs):
    torch.manual_seed(SEED)
    model = CNNWithNorm(norm).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
    updates_per_epoch = N_TRAIN // (micro_batch * accum_steps)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=0.05, total_steps=updates_per_epoch * epochs)
    sched_steps = 0
    for ep in range(epochs):
        torch.manual_seed(SEED + 100 + ep)
        _, n_up = train_with_accumulation(model, opt, X_train, y_train, micro_batch, accum_steps, sched, updates_per_epoch)
        sched_steps += n_up
    vl, va = evaluate(model, X_val, y_val)
    flat = torch.cat([p.detach().flatten() for p in model.parameters()])
    return va, sched_steps, flat


EPOCHS_EX2 = 1 if FAST_RUN else 2
for norm in ("bn", "gn"):
    va_big, steps_big, w_big = accumulation_run(norm, 128, 1, EPOCHS_EX2)
    va_acc, steps_acc, w_acc = accumulation_run(norm, 32, 4, EPOCHS_EX2)
    print(f"{norm.upper()} | batch 128: acc {va_big:.4f}, scheduler steps {steps_big} | 32x4 accumulated: acc {va_acc:.4f}, "
          f"scheduler steps {steps_acc} | max |w_big - w_acc| = {(w_big - w_acc).abs().max():.2e}")

With BatchNorm the final weights differ by a clearly non-zero amount and the accuracies differ by a few
tenths of a percent (sometimes more); with GroupNorm the weights agree to rounding error. In practice
BatchNorm plus accumulation is still widely used, because the micro-batch statistics are merely
noisier rather than wrong, but micro-batches below about 16 samples degrade BatchNorm noticeably, and
architectures designed for small per-device batches (detection, segmentation, large images) use
GroupNorm or synchronized BatchNorm (`nn.SyncBatchNorm`, which all-reduces the statistics across
DDP ranks) for this reason.

### Exercise 3: a mixed-precision ablation

Using `train_steps_amp` from Section 3.3, run the MLP *and* the CNN under every precision policy your
device supports, and report time per step and mean loss. Then, for one batch, compare the gradient of
the CNN computed in float32 with the gradient computed under autocast (bfloat16 on CPU, float16 with a
scaler on GPU): report the relative difference $\|g_{\text{amp}} - g_{32}\| / \|g_{32}\|$ and comment
on whether it is a concern.

**Solution to Exercise 3**

The benchmark loop is a direct reuse of the helper. For the gradient comparison we run the same model
on the same batch twice; under float16 the scaler multiplies the loss, so we must call
`scaler.unscale_(opt)` before reading the gradients (or simply divide by the scale). The relative
difference is typically $10^{-3}$ to $10^{-2}$, far below the differences between two random
mini-batches, which is why mixed precision does not change the trajectory of SGD in any meaningful
way: the gradient noise from sampling dwarfs the rounding noise.

In [ ]:
STEPS_EX3 = 10 if FAST_RUN else 60
if device.type == "cuda":
    policies = [("float32", None), ("float16+scaler", torch.float16)] + ([("bfloat16", torch.bfloat16)] if torch.cuda.is_bf16_supported() else [])
else:
    policies = [("float32", None), ("bfloat16", torch.bfloat16)]

for model_name, fn, bs in [("MLP", lambda: MLP(hidden=512, depth=3), 128), ("CNN", lambda: SmallCNN(width=32), 64)]:
    for pol_name, dt in policies:
        try:
            per_step, mean_loss, _ = train_steps_amp(fn, STEPS_EX3, dt, batch_size=bs)
            print(f"{model_name} {pol_name:15s} {per_step * 1e3:8.2f} ms/step | mean loss {mean_loss:.3f}")
        except Exception as e:
            print(f"{model_name} {pol_name:15s} unsupported here ({type(e).__name__})")

# gradient comparison on one batch
torch.manual_seed(SEED)
cmp_model = SmallCNN().to(device)
xb, yb = X_train[:64], y_train[:64]
cmp_model.train()
cmp_model.zero_grad(set_to_none=True)
F.cross_entropy(cmp_model(xb), yb).backward()
g32 = torch.cat([p.grad.flatten() for p in cmp_model.parameters()]).clone()

amp_dt = torch.float16 if device.type == "cuda" else torch.bfloat16
cmp_opt = torch.optim.SGD(cmp_model.parameters(), lr=0.1)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
cmp_model.zero_grad(set_to_none=True)
try:
    with torch.amp.autocast(device_type=device.type, dtype=amp_dt):
        loss_amp = F.cross_entropy(cmp_model(xb), yb)
    scaler.scale(loss_amp).backward()
    scaler.unscale_(cmp_opt)                                   # gradients back to the true scale
    g_amp = torch.cat([p.grad.flatten() for p in cmp_model.parameters()]).float()
    print(f"\nrelative gradient difference ({amp_dt} vs float32): {((g_amp - g32).norm() / g32.norm()).item():.2e}")
except Exception as e:
    print("autocast gradient comparison unsupported here:", type(e).__name__)

### Exercise 4: extend the Optuna study to architecture choices

Extend the objective of Section 6 with two architectural hyperparameters, the depth (1 to 3 hidden
layers) and the activation (ReLU or GELU), and make the learning-rate range *conditional* on the
optimizer (SGD: $10^{-2}$ to $3 \times 10^{-1}$; AdamW: $10^{-4}$ to $10^{-2}$). Use the multivariate
TPE sampler, run a short study, and report the best configuration and the parameter importances.

**Solution to Exercise 4**

Conditional ranges are just Python: which `suggest_float` call is issued depends on the value already
drawn for `optimizer`. We give the two branches different parameter names (`lr_sgd`, `lr_adamw`) so
that every name has one fixed distribution; Optuna also allows reusing a single name with different
bounds in different trials (a *dynamic* search space), but distinct names keep the sampler's
search-space bookkeeping and the importance analysis unambiguous. The multivariate TPE
(`TPESampler(multivariate=True)`) models the joint density of all parameters instead of one density
per parameter, which helps precisely when parameters interact, as learning rate and optimizer do;
`group=True` lets it decompose a conditional space such as ours into the groups of parameters that
occur together. Both options are still marked experimental in Optuna, so we silence that warning.

In [ ]:
class MLPFlex(nn.Module):
    """MLP with a configurable activation, for the extended study."""

    def __init__(self, hidden=256, depth=2, activation="relu", dropout=0.0):
        super().__init__()
        act = nn.ReLU if activation == "relu" else nn.GELU
        layers, d_in = [], 28 * 28
        for _ in range(depth):
            layers += [nn.Linear(d_in, hidden), act(), nn.Dropout(dropout)]
            d_in = hidden
        layers.append(nn.Linear(d_in, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.flatten(1))


def objective_arch(trial):
    optimizer = trial.suggest_categorical("optimizer", ["sgd", "adamw"])
    if optimizer == "sgd":
        lr = trial.suggest_float("lr_sgd", 1e-2, 3e-1, log=True)       # conditional range, own name
    else:
        lr = trial.suggest_float("lr_adamw", 1e-4, 1e-2, log=True)
    hidden = trial.suggest_categorical("hidden", [128, 256, 512])
    depth = trial.suggest_int("depth", 1, 3)
    activation = trial.suggest_categorical("activation", ["relu", "gelu"])
    dropout = trial.suggest_float("dropout", 0.0, 0.4)

    torch.manual_seed(SEED)
    model = MLPFlex(hidden, depth, activation, dropout).to(device)
    opt = (torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9) if optimizer == "sgd"
           else torch.optim.AdamW(model.parameters(), lr=lr))
    report_every = max(1, STEPS_PER_EPOCH // REPORTS_PER_EPOCH)
    step = 0
    for epoch in range(EPOCHS_PER_TRIAL):
        model.train()
        for xb, yb in iterate_minibatches(X_opt, y_opt, 128):
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            step += 1
            if step % report_every == 0:
                _, acc = evaluate(model, X_opt_val, y_opt_val)
                trial.report(acc, step)
                if trial.should_prune():
                    raise optuna.TrialPruned()
                model.train()
    return evaluate(model, X_opt_val, y_opt_val)[1]


if HAVE_OPTUNA:
    import warnings
    warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
    N_TRIALS_EX4 = 6 if FAST_RUN else 30
    study_arch = optuna.create_study(direction="maximize",
                                     sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True, group=True),
                                     pruner=optuna.pruners.MedianPruner(n_startup_trials=3))
    study_arch.optimize(objective_arch, n_trials=N_TRIALS_EX4, show_progress_bar=False)
    done = [t for t in study_arch.trials if t.state == TrialState.COMPLETE]
    print(f"{len(study_arch.trials)} trials, {len(done)} completed | best acc {study_arch.best_value:.4f}")
    print("best params:", study_arch.best_params)
    try:
        for k, v in optuna.importance.get_param_importances(study_arch).items():
            print(f"  importance {k:12s} {v:.3f}")
    except Exception as e:
        print("importance evaluation unavailable:", type(e).__name__)
else:
    print("optuna is not installed; skipping the extended study.")

With only a few trials the importances are unreliable, and the two conditional learning-rate parameters
are normally missing from the importance table altogether, because the evaluator only considers
parameters that are present in every completed trial (their effect is visible in the best parameters
instead). The study structure is what matters: the conditional learning-rate range removes the wasted
trials of Section 6 (SGD at $10^{-4}$, AdamW at $10^{-1}$), and the architecture parameters can now be
compared on equal footing with the optimizer ones. In a real project this study would run for a hundred trials in a SQLite-backed study from
several worker processes.

### Exercise 5 (challenge): simulate data parallelism on one device

Implement data-parallel training by hand: keep $K = 4$ replicas of the MLP on the single device, give
each replica one quarter of every batch of 128, compute the local gradients, "all-reduce" them by
averaging across replicas, copy the averaged gradient into every replica and let each replica's
optimizer step. (a) Verify after several steps that all replicas are still bitwise identical.
(b) Verify that the replicas match a single model trained on the full batch of 128 with the same seed.
(c) Compute the bytes each replica would send over the network per step under a ring all-reduce, and
the ratio of communication to computation time at an assumed interconnect bandwidth of 10 GB/s and
the measured step time.

**Solution to Exercise 5**

The essential invariant of data parallelism is that all replicas execute the *same* update. Averaging
the gradients and copying the average into every replica reproduces what NCCL's all-reduce does; since
the replicas start from the same weights and receive the same gradient, they stay identical without
any parameter communication. Because the loss is a mean over each shard and the shards have equal
sizes, the averaged gradient equals the gradient of the mean loss over the whole batch, so the
replicas also match the single-model run. The ring all-reduce moves $2(K-1)/K$ times the gradient size
through every replica's link per step.

In [ ]:
K_REPL = 4


def make_replicas(k):
    torch.manual_seed(SEED)
    base = MLP().to(device)
    replicas = [copy.deepcopy(base) for _ in range(k)]
    opts = [torch.optim.SGD(r.parameters(), lr=0.05, momentum=0.9) for r in replicas]
    return replicas, opts


def data_parallel_step(replicas, opts, xb, yb):
    """One synchronous data-parallel step with a simulated all-reduce (average of gradients)."""
    k = len(replicas)
    shards = torch.chunk(torch.arange(xb.shape[0], device=xb.device), k)
    for r, o, idx in zip(replicas, opts, shards):
        r.train()
        o.zero_grad(set_to_none=True)
        F.cross_entropy(r(xb[idx]), yb[idx]).backward()          # local gradient on the local shard
    for params in zip(*[r.parameters() for r in replicas]):     # "all-reduce": average per parameter
        g = torch.stack([p.grad for p in params]).mean(0)
        for p in params:
            p.grad.copy_(g)
    for o in opts:
        o.step()                                                 # identical update everywhere


replicas, opts = make_replicas(K_REPL)
torch.manual_seed(SEED)
single = MLP().to(device)
single_opt = torch.optim.SGD(single.parameters(), lr=0.05, momentum=0.9)

N_DP_STEPS = 20 if FAST_RUN else 100
torch.manual_seed(SEED + 1)
batches = iterate_minibatches(X_train, y_train, 128)
sync(); t0 = time.perf_counter()
for step in range(N_DP_STEPS):
    xb, yb = next(batches)
    data_parallel_step(replicas, opts, xb, yb)
    single.train()
    single_opt.zero_grad(set_to_none=True)
    F.cross_entropy(single(xb), yb).backward()
    single_opt.step()
sync(); t_step = (time.perf_counter() - t0) / N_DP_STEPS

flat = lambda m: torch.cat([p.detach().flatten() for p in m.parameters()])
w0 = flat(replicas[0])
print(f"(a) max difference between replicas: {max((flat(r) - w0).abs().max().item() for r in replicas[1:]):.1e}")
print(f"(b) max difference replica vs single large-batch model: {(w0 - flat(single)).abs().max().item():.1e}")
grad_bytes = count_params(single) * 4
ring_bytes = 2 * (K_REPL - 1) / K_REPL * grad_bytes
t_comm = ring_bytes / 10e9
print(f"(c) gradient size {grad_bytes / MB:.2f} MB; ring all-reduce traffic per replica per step {ring_bytes / MB:.2f} MB "
      f"-> {t_comm * 1e3:.2f} ms at 10 GB/s vs measured step {t_step * 1e3:.1f} ms (K replicas, this device); "
      f"communication/compute = {t_comm / t_step:.2f}")

The replicas remain bitwise identical (difference exactly zero) and agree with the single large-batch
model to rounding error, which is the whole correctness argument of DDP in one experiment. The
communication estimate shows why data parallelism scales well for models with a high compute-to-
parameter ratio (CNNs) and worse for models dominated by embeddings, and why bucketing the all-reduce
behind the backward pass matters: without overlap, the communication time is added to every step.

## Summary

- Time GPU code only after `torch.cuda.synchronize()`, after warm-up, and with medians; `.item()`
  and `.cpu()` inside the loop are hidden synchronizations. `torch.profiler` tells you whether a step
  is data-, launch-, memory- or compute-bound.
- Per-sample `DataLoader` pipelines cost tens of microseconds per sample in Python; small datasets
  belong in device memory as tensors, large ones need workers, pinned memory and cheap transforms.
- Training memory is $16$ bytes per parameter with Adam (also in mixed precision) plus batch size ×
  activation elements × bytes per element; `saved_tensors_hooks` and `max_memory_allocated` measure it.
- float16 has range problems (needs `GradScaler` loss scaling), bfloat16 has precision problems that
  SGD tolerates; use `torch.amp.autocast` with bfloat16 where the hardware supports it, never
  `model.half()`.
- Gradient accumulation reproduces a large batch exactly (divide the loss by $K$, step the scheduler
  per update) except for BatchNorm statistics; activation checkpointing trades about a third more
  compute for a several-fold reduction of activation memory.
- `torch.compile` fuses operators and removes launch overhead; it pays off for long runs with static
  shapes and needs a C++ toolchain. TF32 (`set_float32_matmul_precision("high")`) is a free 2–4× on
  Ampere+ matmuls.
- Optuna's TPE sampler with a `MedianPruner` finds good configurations in a few dozen trials; read
  the history, importance and parallel-coordinate plots, and tune on validation data only.
- DDP averages gradients with bucketed all-reduces overlapped with the backward pass; FSDP/ZeRO shard
  the training state so that memory per device falls as $16/N$ bytes per parameter.
- Training costs $C \approx 6ND$ FLOPs; compute-optimal models use about 20 tokens per parameter,
  and deployed models are deliberately trained past that point.
- Every experiment is a serialized config plus an append-only metrics log; ablate one field at a time
  and use several seeds before believing a difference.

## Further reading

- Micikevicius et al., 2018, *Mixed Precision Training* (ICLR) — the original float16 training recipe with loss scaling and master weights.
- Chen et al., 2016, *Training Deep Nets with Sublinear Memory Cost* — activation checkpointing and the $O(\sqrt{L})$ argument.
- Korthikanti et al., 2022, *Reducing Activation Recomputation in Large Transformer Models* — activation memory formulas for Transformers and selective checkpointing.
- Goyal et al., 2017, *Accurate, Large Minibatch SGD: Training ImageNet in 1 Hour* — the linear scaling rule and warm-up.
- Li et al., 2020, *PyTorch Distributed: Experiences on Accelerating Data Parallel Training* (VLDB) — the design of DistributedDataParallel, bucketing and overlap.
- Rajbhandari et al., 2020, *ZeRO: Memory Optimizations Toward Training Trillion Parameter Models* — the three sharding stages behind FSDP.
- Dettmers et al., 2022, *8-bit Optimizers via Block-wise Quantization* (ICLR).
- Ansel et al., 2024, *PyTorch 2: Faster Machine Learning Through Dynamic Python Bytecode Transformation and Graph Compilation* (ASPLOS) — TorchDynamo and TorchInductor.
- Akiba et al., 2019, *Optuna: A Next-generation Hyperparameter Optimization Framework* (KDD); Bergstra et al., 2011, *Algorithms for Hyper-Parameter Optimization* (the TPE sampler).
- Kaplan et al., 2020, *Scaling Laws for Neural Language Models*; Hoffmann et al., 2022, *Training Compute-Optimal Large Language Models* (Chinchilla).
- PyTorch documentation: automatic mixed precision examples (https://docs.pytorch.org/docs/stable/notes/amp_examples.html), DDP notes (https://docs.pytorch.org/docs/stable/notes/ddp.html), `torch.compile` (https://docs.pytorch.org/docs/stable/torch.compiler.html), and the profiler recipe in the official tutorials.